# Guia: Black-Scholes explicável e DQN para o ponto de entrada da proteção com put

Este notebook é um **guia de estudo**. Ele mostra, com dados reais, cada conta que existe por trás de duas peças do seu mestrado:

- **Parte A: Black-Scholes.** De onde vem a fórmula da put, o que cada termo significa, como ela se compara com o preço real do mercado e o que ela consegue (e não consegue) dizer.
- **Parte B: Deep Q-Network.** Como transformar "quando comprar a proteção" num problema de aprendizado por reforço, como o agente aprende, como avaliá-lo de forma honesta e como abrir a "caixa-preta" para entender as decisões.

### Como cada seção está organizada
1. **O que vamos fazer**: a ideia em linguagem simples e as fórmulas.
2. **Código**: funções curtas, com nomes que dizem o que fazem.
3. **Leitura do resultado**: um relatório gerado com os números reais da execução, explicando o que eles significam.
4. **Para julgar artigos**: o que conferir quando um artigo usa a mesma técnica.

### Mapa
| Parte | Seção | Pergunta que responde |
|---|---|---|
| 0 | Dados | O que o Yahoo Finance entrega e com que qualidade? |
| A1 | A opção como objeto | O que a put paga e como ela protege a ação? |
| A2 | Hipóteses do modelo | Os retornos reais se comportam como o Black-Scholes supõe? |
| A3 | A fórmula, peça por peça | O que cada termo da fórmula da put representa? |
| A4 | Por que o retorno esperado some | Por que o preço da put não depende de a ação "ir subir"? |
| A5 | Prova por Monte Carlo | A fórmula é mesmo a média dos payoffs descontados? |
| A6 | Gregas e Delta × preço | Como o preço da put reage a cada variável? |
| A7 | Volatilidade implícita e smile | Que volatilidade o mercado está cobrando, strike a strike? |
| A8 | Paridade put-call | Os preços reais de calls e puts são coerentes entre si? |
| A9 | Caro ou barato? | A put está cara em relação ao risco previsto (GARCH)? |
| B1 | Dados para a DQN | Que sinais o agente enxerga a cada dia? |
| B2 | O problema como MDP | Estado, ação, recompensa: a formulação formal. |
| B3 | Recompensa | O que o agente tenta maximizar, em números? |
| B4 | Memória do agente | Por que guardar e sortear experiências? |
| B5 | A rede Q | Como a rede calcula e aprende (com verificação do gradiente). |
| B6 | Política | Como o agente equilibra explorar e aproveitar? |
| B7 | Treinamento | O agente aprendeu? Curvas e várias sementes. |
| B8 | Avaliação passo a passo | A DQN supera regras simples fora da amostra? |
| B9 | Explicabilidade | Por que o agente comprou em determinado dia? |
| B10 | Relatório final | Resumo e checklist para julgar artigos. |

### Fonte dos dados
A leitura dos dados é a **mesma do `put_de_protecao.ipynb`**: Yahoo Finance para preços, dividendos, juros (`^IRX`) e a **cadeia real de opções** de hoje. Dois limites que você precisa ter em mente:
- O Yahoo publica opções de ativos dos **EUA**, que são de **estilo americano** (podem ser exercidas antes do vencimento). O Black-Scholes é para opções **europeias**; a seção A7 mede o tamanho dessa diferença.
- O Yahoo **não guarda o histórico** de preços de opções. Por isso, na Parte B, o custo histórico da put é uma **estimativa calibrada** ao mercado de hoje, e o notebook diz isso em todos os lugares onde ela aparece. Para usar preços históricos reais (COTAHIST da B3 ou outra base), basta trocar uma única função, `estimate_historical_put_premiums`.

> Material de estudo. Nada aqui é recomendação de investimento.

## Configuração

Todos os parâmetros ficam em `GuideSettings`. Para estudar outro ativo, troque `ticker`. Para treinar a DQN com a outra recompensa, troque `reward_type`.

In [ ]:
import copy
import logging
import warnings
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import Markdown, display
from matplotlib.ticker import PercentFormatter
from scipy import optimize, signal, stats
from scipy.special import gammaln


@dataclass(frozen=True)
class GuideSettings:
    ticker: str = "AAPL"                     # ativo estudado (formato Yahoo; opções só para ativos dos EUA)
    history_start_date: str = "2012-01-01"

    # Escolha da put real de hoje (Parte A)
    target_protection_days: int = 45         # prazo desejado da proteção, em dias corridos
    minimum_protection_days: int = 14
    strike_range_vs_price: tuple = (0.80, 1.10)
    minimum_open_interest: int = 50
    protection_moneyness: float = 0.95       # strike ÷ preço da put usada como exemplo e na Parte B

    # Cálculos numéricos da Parte A
    binomial_steps: int = 300                # árvore binomial (opção americana)
    binomial_steps_for_iv: int = 200         # árvore usada dentro da busca da volatilidade implícita
    hedge_simulation_paths: int = 5_000      # simulação do delta hedge (seção A4)

    # DQN (Parte B)
    entry_window_days: int = 10              # dias úteis que você aceita esperar para entrar
    reward_type: str = "resultado_protegido" # "resultado_protegido" ou "economia_na_entrada"
    dqn_training_rounds: int = 150           # cada rodada roda 64 episódios em paralelo
    dqn_hidden_units: int = 64
    dqn_seeds: tuple = (0, 1, 2, 3, 4)       # várias sementes: um resultado só vale se aparecer em quase todas
    test_period_fraction: float = 0.30       # fração final do histórico reservada para teste

    random_seed: int = 42


SETTINGS = GuideSettings()

warnings.filterwarnings("ignore")
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

COLORS = {"stock": "#2a78d6", "protected": "#eb6834", "accent": "#1baf7a", "warning": "#eda100",
          "ink": "#52514e", "ink_strong": "#0b0b0b", "grid": "#e6e5e1", "surface": "#fcfcfb",
          "loss": "#d03b3b", "muted": "#a3a29c", "purple": "#7b5cc4"}
plt.rcParams.update({
    "figure.facecolor": COLORS["surface"], "axes.facecolor": COLORS["surface"], "savefig.facecolor": COLORS["surface"],
    "axes.edgecolor": COLORS["grid"], "axes.labelcolor": COLORS["ink"], "xtick.color": COLORS["ink"],
    "ytick.color": COLORS["ink"], "text.color": COLORS["ink_strong"], "axes.grid": True, "grid.color": COLORS["grid"],
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2, "figure.figsize": (10, 4.4),
    "font.size": 10, "axes.titleweight": "bold", "axes.titlesize": 11, "legend.frameon": False,
})
TODAY = pd.Timestamp.today().normalize()
TRADING_DAYS_PER_YEAR = 252


def report(title: str, findings: list) -> None:
    # Relatório explicável: cada achado numérico vira uma frase em português.
    lines = [f"#### Leitura do resultado: {title}"] + [f"- {finding}" for finding in findings]
    display(Markdown("\n".join(lines)))


SETTINGS

## 0. Dados: o que o Yahoo Finance entrega

**O que vamos fazer.** Usar as mesmas funções de leitura do `put_de_protecao.ipynb`, com duas adições: a cadeia de **calls** (para a paridade put-call, seção A8) e o **histórico da taxa de juros** (para a Parte B). Cada função busca uma informação e, se o dado não existir, lança `MarketDataError` com o motivo.

In [ ]:
class MarketDataError(Exception):
    # Dado indisponível no Yahoo Finance.
    pass


def to_naive_dates(index) -> pd.DatetimeIndex:
    # Remove o fuso horário mantendo a data local do pregão.
    dates = pd.to_datetime(index)
    return dates.tz_localize(None) if dates.tz is not None else dates


def download_price_history(ticker: str, start_date: str) -> pd.Series:
    # Fechamentos diários ajustados por dividendos e desdobramentos.
    history = yf.Ticker(ticker).history(start=start_date, auto_adjust=True)
    if history is None or history.empty:
        raise MarketDataError("sem histórico de preços no Yahoo")
    closes = history["Close"].dropna()
    closes.index = to_naive_dates(closes.index)
    return closes[closes > 0]


def get_current_stock_price(ticker: str, fallback_price: float) -> float:
    # Último preço negociado (fast_info); se indisponível, o último fechamento.
    try:
        last_price = float(yf.Ticker(ticker).fast_info["lastPrice"])
        return last_price if last_price > 0 else fallback_price
    except Exception:
        return fallback_price


def get_trailing_dividend_yield(ticker: str, current_price: float) -> float:
    # Dividendos pagos nos últimos 12 meses ÷ preço atual.
    dividends = yf.Ticker(ticker).dividends
    if dividends is None or dividends.empty:
        return 0.0
    dividends.index = to_naive_dates(dividends.index)
    last_year_total = dividends[dividends.index >= TODAY - pd.Timedelta(days=365)].sum()
    return float(last_year_total / current_price)


def get_us_risk_free_rate() -> float:
    # Taxa do Treasury de 13 semanas (^IRX), em fração ao ano.
    history = yf.Ticker("^IRX").history(period="1mo")
    if history is None or history.empty:
        raise MarketDataError("sem taxa ^IRX no Yahoo")
    return float(history["Close"].dropna().iloc[-1]) / 100


def download_risk_free_history(start_date: str) -> pd.Series:
    # Histórico diário da mesma taxa (^IRX), usado na Parte B.
    history = yf.Ticker("^IRX").history(start=start_date)
    if history is None or history.empty:
        raise MarketDataError("sem histórico da taxa ^IRX")
    rates = history["Close"].dropna() / 100
    rates.index = to_naive_dates(rates.index)
    return rates


def list_option_expirations(ticker: str) -> list:
    # Vencimentos de opções disponíveis no Yahoo.
    expirations = yf.Ticker(ticker).options
    if not expirations:
        raise MarketDataError("o Yahoo não tem opções para este ativo")
    return [pd.Timestamp(date) for date in expirations]


def choose_expiration(expirations: list, settings: GuideSettings) -> pd.Timestamp:
    # Vencimento mais próximo do prazo desejado, respeitando o prazo mínimo.
    long_enough = [date for date in expirations if (date - TODAY).days >= settings.minimum_protection_days]
    if not long_enough:
        raise MarketDataError("nenhum vencimento com prazo mínimo")
    return min(long_enough, key=lambda date: abs((date - TODAY).days - settings.target_protection_days))


def download_option_chain(ticker: str, expiration: pd.Timestamp) -> tuple:
    # Calls e puts de um vencimento, como o Yahoo publica.
    chain = yf.Ticker(ticker).option_chain(expiration.strftime("%Y-%m-%d"))
    if chain.puts.empty:
        raise MarketDataError(f"sem puts para o vencimento {expiration.date()}")
    return chain.calls.copy(), chain.puts.copy()


def prepare_quotes(raw_options: pd.DataFrame, stock_price: float, settings: GuideSettings) -> pd.DataFrame:
    # Mantém strikes na faixa e com contratos em aberto; define o preço de referência (mid) e o de compra (ask).
    options = raw_options.copy()
    lowest_strike, highest_strike = (stock_price * bound for bound in settings.strike_range_vs_price)
    options = options[(options["strike"] >= lowest_strike) & (options["strike"] <= highest_strike)]
    has_live_quote = (options["bid"] > 0) & (options["ask"] > 0)
    options["has_live_quote"] = has_live_quote
    options["mid"] = np.where(has_live_quote, (options["bid"] + options["ask"]) / 2, options["lastPrice"])
    options["purchase_price"] = np.where(has_live_quote, options["ask"], options["lastPrice"])
    options["spread_pct"] = np.where(has_live_quote, (options["ask"] - options["bid"]) / options["mid"], np.nan)
    options = options[(options["openInterest"].fillna(0) >= settings.minimum_open_interest) & (options["mid"] > 0)]
    if options.empty:
        raise MarketDataError("nenhuma opção líquida na faixa de strikes")
    columns = ["contractSymbol", "strike", "bid", "ask", "lastPrice", "mid", "purchase_price", "has_live_quote",
               "spread_pct", "openInterest", "volume", "impliedVolatility"]
    return options[columns].sort_values("strike").reset_index(drop=True)

In [ ]:
TICKER = SETTINGS.ticker
CLOSES = download_price_history(TICKER, SETTINGS.history_start_date)
SPOT = get_current_stock_price(TICKER, float(CLOSES.iloc[-1]))
DIVIDEND_YIELD = get_trailing_dividend_yield(TICKER, SPOT)
RISK_FREE_RATE = get_us_risk_free_rate()
RATE_HISTORY = download_risk_free_history(SETTINGS.history_start_date)
EXPIRATIONS = list_option_expirations(TICKER)
EXPIRATION = choose_expiration(EXPIRATIONS, SETTINGS)
DAYS_TO_EXPIRATION = max((EXPIRATION - TODAY).days, 1)
T_YEARS = DAYS_TO_EXPIRATION / 365
RAW_CALLS, RAW_PUTS = download_option_chain(TICKER, EXPIRATION)

print(f"{TICKER}: {len(CLOSES)} fechamentos de {CLOSES.index[0].date()} a {CLOSES.index[-1].date()}")
print(f"Preço atual {SPOT:,.2f} · dividend yield {DIVIDEND_YIELD:.2%} · juros (^IRX) {RISK_FREE_RATE:.2%} ao ano")
print(f"{len(EXPIRATIONS)} vencimentos disponíveis; escolhido {EXPIRATION.date()} ({DAYS_TO_EXPIRATION} dias)")
display(RAW_PUTS.head(8))

### O que significa cada coluna da cadeia de opções

| Coluna | Significado | Como o notebook usa |
|---|---|---|
| `contractSymbol` | Código do contrato (ativo, vencimento AAMMDD, P de put, strike × 1000) | identificação |
| `strike` | Preço de exercício: o valor pelo qual a put garante a venda | entrada da fórmula |
| `bid` | Maior oferta de **compra** no livro: o que você recebe se vender a opção | referência |
| `ask` | Menor oferta de **venda**: o que você **paga** para comprar a put | **custo real da proteção** |
| `lastPrice` | Preço do último negócio, que pode ser antigo | usado só se bid/ask estiverem zerados |
| `openInterest` | Contratos em aberto: indica liquidez | filtro de liquidez |
| `volume` | Contratos negociados hoje | referência |
| `impliedVolatility` | Volatilidade implícita **calculada pelo Yahoo** (método não documentado) | só para comparar com a nossa (A7) |

O **mid** = (bid + ask) ÷ 2 é a melhor estimativa do "preço justo" do mercado; o **ask** é o que você realmente paga. A diferença entre os dois (o *spread*) é um custo de transação.

In [ ]:
PUTS = prepare_quotes(RAW_PUTS, SPOT, SETTINGS)
try:
    CALLS = prepare_quotes(RAW_CALLS, SPOT, SETTINGS)
except MarketDataError:
    CALLS = None

puts_in_range = RAW_PUTS[(RAW_PUTS["strike"] >= SPOT * SETTINGS.strike_range_vs_price[0])
                         & (RAW_PUTS["strike"] <= SPOT * SETTINGS.strike_range_vs_price[1])]
live_share = PUTS["has_live_quote"].mean()
recent_close_gap = SPOT / CLOSES.iloc[-1] - 1
findings = [
    f"O histórico tem **{len(CLOSES)} pregões**. O preço atual difere do último fechamento em {recent_close_gap:+.2%} "
    "(diferença normal durante o pregão; se for grande fora do pregão, desconfie do dado).",
    f"A cadeia do vencimento {EXPIRATION.date()} tem **{len(RAW_PUTS)} puts**; {len(puts_in_range)} ficam na faixa de strikes "
    f"estudada ({SETTINGS.strike_range_vs_price[0]:.0%} a {SETTINGS.strike_range_vs_price[1]:.0%} do preço) e "
    f"**{len(PUTS)}** passam no filtro de liquidez (pelo menos {SETTINGS.minimum_open_interest} contratos em aberto).",
    f"{live_share:.0%} das puts líquidas têm bid e ask ao vivo." + (
        "" if live_share > 0.5 else " **Atenção:** com poucas cotações ao vivo (mercado fechado?), o notebook usa o último negócio, "
        "que pode estar desatualizado. Rode durante o pregão para resultados mais confiáveis."),
]
if PUTS["spread_pct"].notna().any():
    findings.append(f"O spread mediano é **{PUTS['spread_pct'].median():.1%}** do mid: esse é o custo só de entrar e sair da put. "
                    "Spreads acima de 10% tornam qualquer \"vantagem\" de modelo difícil de capturar na prática.")
findings.append("Calls disponíveis para a paridade put-call (A8): " + ("sim." if CALLS is not None else "**não**, a seção A8 será pulada."))
report("qualidade dos dados", findings)

> **Para julgar artigos.** Verifique (1) se o artigo usa **bid/ask** ou só o último negócio, que pode ser antigo; (2) se filtra liquidez (open interest, volume, spread); (3) se diz o **horário** das cotações. Resultados com preços de fechamento de opções pouco negociadas costumam mostrar "oportunidades" que não existem na prática.

---
# Parte A — Black-Scholes explicável

## A1. A opção como objeto e o payoff no vencimento

**O que vamos fazer.** Representar a put como um objeto com três informações (ativo, strike, vencimento) e calcular o que ela paga no vencimento:

$$\text{payoff da put} = \max(K - S_T,\ 0)$$

onde $K$ é a strike e $S_T$ é o preço da ação no vencimento. Se a ação terminar abaixo da strike, a put paga a diferença; se terminar acima, não paga nada.

Somando a put à ação, o resultado da **posição protegida** por ação é:

$$\underbrace{S_T - S_0}_{\text{ação}} + \underbrace{\max(K - S_T, 0) - P}_{\text{put}} \;\ge\; K - S_0 - P$$

O lado direito é o **piso**: a perda máxima, conhecida no dia da compra. Neste exemplo usamos a put real mais próxima de `protection_moneyness` × preço, pelo **ask** (o preço que você pagaria). Taxas e corretagem ficam de fora para deixar as contas limpas.

In [ ]:
@dataclass(frozen=True)
class PutOption:
    underlying: str
    strike: float
    expiration: pd.Timestamp

    def payoff_at_expiration(self, prices_at_expiration) -> np.ndarray:
        # O que a put paga no vencimento: max(K − S_T, 0).
        return np.maximum(self.strike - np.asarray(prices_at_expiration, float), 0.0)

    def years_to_expiration(self, valuation_date: pd.Timestamp = TODAY) -> float:
        return max((self.expiration - valuation_date).days, 1) / 365


example_row = PUTS.iloc[(PUTS["strike"] - SETTINGS.protection_moneyness * SPOT).abs().argmin()]
EXAMPLE_PUT = PutOption(TICKER, float(example_row["strike"]), EXPIRATION)
EXAMPLE_PREMIUM = float(example_row["purchase_price"])
EXAMPLE_MID = float(example_row["mid"])

price_grid = np.linspace(SPOT * 0.6, SPOT * 1.4, 400)
stock_profit = price_grid - SPOT
put_profit = EXAMPLE_PUT.payoff_at_expiration(price_grid) - EXAMPLE_PREMIUM
protected_profit = stock_profit + put_profit
max_loss_per_share = SPOT + EXAMPLE_PREMIUM - EXAMPLE_PUT.strike

figure, axis = plt.subplots(figsize=(10, 4.6))
axis.plot(price_grid, stock_profit, color=COLORS["stock"], label="só a ação")
axis.plot(price_grid, put_profit, color=COLORS["muted"], lw=1.5, ls="--", label="só a put (payoff − prêmio)")
axis.plot(price_grid, protected_profit, color=COLORS["protected"], lw=2.5, label="ação + put")
axis.axhline(0, color=COLORS["ink"], lw=1)
axis.axhline(-max_loss_per_share, color=COLORS["loss"], lw=1, ls=":")
axis.annotate(f"piso: perda máxima {max_loss_per_share:,.2f}", (price_grid[0], -max_loss_per_share),
              xytext=(4, -14), textcoords="offset points", color=COLORS["loss"])
axis.axvline(EXAMPLE_PUT.strike, color=COLORS["ink"], lw=1, ls=":")
axis.annotate(f"strike {EXAMPLE_PUT.strike:,.2f}", (EXAMPLE_PUT.strike, axis.get_ylim()[1]), xytext=(4, -14),
              textcoords="offset points", color=COLORS["ink"])
axis.set_xlabel("preço da ação no vencimento")
axis.set_ylabel("resultado por ação")
axis.set_title(f"{TICKER}: payoff no vencimento da ação, da put e da posição protegida")
axis.legend(loc="upper left")
plt.show()

report("payoff da proteção", [
    f"Ação a **{SPOT:,.2f}**; put de strike **{EXAMPLE_PUT.strike:,.2f}** ({EXAMPLE_PUT.strike / SPOT:.0%} do preço), "
    f"vencimento {EXPIRATION.date()} ({DAYS_TO_EXPIRATION} dias). Prêmio pago (ask): **{EXAMPLE_PREMIUM:,.2f}** "
    f"({EXAMPLE_PREMIUM / SPOT:.2%} do preço da ação).",
    f"Perda máxima = preço + prêmio − strike = **{max_loss_per_share:,.2f} por ação** "
    f"({max_loss_per_share / (SPOT + EXAMPLE_PREMIUM):.1%} do valor investido), qualquer que seja a queda.",
    f"Break-even: a ação precisa terminar acima de **{SPOT + EXAMPLE_PREMIUM:,.2f}** ({EXAMPLE_PREMIUM / SPOT:+.2%}) "
    "para a posição empatar. O prêmio é o preço do seguro e sai do seu retorno em todos os cenários.",
    "A put sozinha (linha tracejada) perde no máximo o prêmio. Somada à ação, ela troca a perna de baixo do gráfico "
    "por uma linha reta: é isso que \"perda máxima conhecida\" significa.",
])

> **Para julgar artigos.** Um artigo sobre proteção com put precisa dizer **qual strike** (em % do preço), **qual prazo**, **qual preço** pagou pela put (ask, mid ou teórico) e **com que frequência** renova a proteção. Os resultados mudam muito com essas escolhas; Israelov (2019) mostra que a mesma estratégia pode parecer boa ou ruim dependendo delas.

## A2. As hipóteses do Black-Scholes contra os dados reais

**O que vamos fazer.** O modelo supõe que o preço segue um **movimento browniano geométrico (GBM)**:

$$dS_t = \mu S_t\,dt + \sigma S_t\,dW_t$$

Em palavras: a cada instante, o retorno tem uma parte previsível ($\mu\,dt$) e uma parte aleatória proporcional à volatilidade $\sigma$, com $dW_t$ normal. Consequências testáveis:
1. Os **log-retornos** diários são **normais** (sem caudas gordas, sem assimetria).
2. A volatilidade $\sigma$ é **constante**: dias agitados não "puxam" outros dias agitados.

Vamos testar as duas hipóteses no histórico real e depois simular o GBM.

In [ ]:
LOG_RETURNS = np.log(CLOSES).diff().dropna()
daily_mean, daily_std = LOG_RETURNS.mean(), LOG_RETURNS.std()
HISTORICAL_VOLATILITY = daily_std * np.sqrt(TRADING_DAYS_PER_YEAR)
RECENT_VOLATILITY = LOG_RETURNS.iloc[-TRADING_DAYS_PER_YEAR:].std() * np.sqrt(TRADING_DAYS_PER_YEAR)
PRICE_DRIFT = daily_mean * TRADING_DAYS_PER_YEAR + HISTORICAL_VOLATILITY ** 2 / 2   # μ do GBM (retorno do preço)

skewness = stats.skew(LOG_RETURNS)
excess_kurtosis = stats.kurtosis(LOG_RETURNS)
jarque_bera_statistic, jarque_bera_p_value = stats.jarque_bera(LOG_RETURNS)
standardized = (LOG_RETURNS - daily_mean) / daily_std
observed_extremes = int((standardized.abs() > 4).sum())
expected_extremes = 2 * stats.norm.sf(4) * len(standardized)


def autocorrelations(series: pd.Series, max_lag: int) -> np.ndarray:
    return np.array([series.autocorr(lag) for lag in range(1, max_lag + 1)])


MAX_LAG = 20
return_autocorrelation = autocorrelations(LOG_RETURNS, MAX_LAG)
squared_return_autocorrelation = autocorrelations(LOG_RETURNS ** 2, MAX_LAG)
significance_band = 1.96 / np.sqrt(len(LOG_RETURNS))

figure, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(standardized, bins=120, density=True, color=COLORS["stock"], alpha=0.6, label="retornos reais")
grid = np.linspace(-6, 6, 300)
axes[0].plot(grid, stats.norm.pdf(grid), color=COLORS["protected"], label="normal (hipótese do BS)")
axes[0].set_xlim(-6, 6); axes[0].set_yscale("log"); axes[0].set_ylim(1e-4, 1)
axes[0].set_title("Distribuição (escala log: caudas visíveis)"); axes[0].legend(); axes[0].set_xlabel("desvios-padrão")
(theoretical_quantiles, ordered_values), _ = stats.probplot(standardized, dist="norm")
axes[1].plot(theoretical_quantiles, ordered_values, ".", color=COLORS["stock"], ms=2)
axes[1].plot([-5, 5], [-5, 5], color=COLORS["protected"], lw=1)
axes[1].set_title("QQ-plot contra a normal"); axes[1].set_xlabel("quantil normal"); axes[1].set_ylabel("quantil observado")
lags = np.arange(1, MAX_LAG + 1)
axes[2].bar(lags - 0.2, return_autocorrelation, width=0.4, color=COLORS["stock"], label="retorno")
axes[2].bar(lags + 0.2, squared_return_autocorrelation, width=0.4, color=COLORS["protected"], label="retorno²")
axes[2].axhspan(-significance_band, significance_band, color=COLORS["grid"])
axes[2].set_title("Autocorrelação por defasagem (dias)"); axes[2].legend(); axes[2].set_xlabel("defasagem")
plt.tight_layout(); plt.show()

clustered_lags = int((np.abs(squared_return_autocorrelation) > significance_band).sum())
report("as hipóteses do modelo se sustentam?", [
    f"Volatilidade anualizada: **{HISTORICAL_VOLATILITY:.1%}** no histórico inteiro e **{RECENT_VOLATILITY:.1%}** "
    "nos últimos 12 meses. Se o σ fosse constante, como o modelo supõe, esses números seriam parecidos.",
    f"Curtose em excesso **{excess_kurtosis:.1f}** (a normal tem 0) e assimetria **{skewness:+.2f}**. "
    f"Teste de Jarque-Bera: p-valor {jarque_bera_p_value:.1e}, "
    + ("**rejeita** a normalidade." if jarque_bera_p_value < 0.05 else "não rejeita a normalidade."),
    f"Movimentos maiores que 4 desvios-padrão: **{observed_extremes}** observados contra **{expected_extremes:.2f}** "
    "esperados pela normal. Isso são as caudas gordas: quedas extremas são muito mais frequentes do que o BS supõe, "
    "e é exatamente nessas quedas que a put paga.",
    f"Os retornos quase não têm autocorrelação (barras azuis dentro da faixa cinza), mas os **retornos ao quadrado** têm "
    f"autocorrelação significativa em **{clustered_lags} de {MAX_LAG}** defasagens: dias agitados vêm em grupos "
    "(agrupamento de volatilidade). Isso viola o σ constante e justifica o GARCH (Bollerslev, 1986) usado em A9.",
    "Conclusão: o BS é um **modelo de referência**, não uma descrição exata. O mercado compensa as falhas dele cobrando "
    "volatilidades diferentes por strike (o smile, seção A7).",
])

### Simulando o GBM: mundo real contra mundo neutro ao risco

Abaixo, simulamos trajetórias com dois drifts:
- **Mundo real:** $\mu$ estimado do histórico ($\mu = $ média dos log-retornos $+\ \sigma^2/2$).
- **Mundo neutro ao risco:** $\mu = r - q$ (juros menos dividendos). É neste mundo que o Black-Scholes calcula preços; a seção A4 mostra **por quê**.

No GBM, $\ln S_T$ é normal, então $S_T$ é **log-normal**:

$$S_T = S_0 \exp\!\Big[\big(\mu - \tfrac{\sigma^2}{2}\big)T + \sigma\sqrt{T}\,Z\Big],\quad Z\sim N(0,1)$$

In [ ]:
def simulate_gbm_paths(start_price: float, drift: float, volatility: float, years: float, steps: int,
                       number_of_paths: int, rng: np.random.Generator) -> np.ndarray:
    # Trajetórias do GBM em passos discretos: cada passo soma um log-retorno normal.
    step_years = years / steps
    shocks = rng.standard_normal((number_of_paths, steps))
    log_steps = (drift - 0.5 * volatility ** 2) * step_years + volatility * np.sqrt(step_years) * shocks
    log_paths = np.hstack([np.zeros((number_of_paths, 1)), np.cumsum(log_steps, axis=1)])
    return start_price * np.exp(log_paths)


def simulate_terminal_prices(start_price: float, drift: float, volatility: float, years: float,
                             number_of_paths: int, rng: np.random.Generator) -> np.ndarray:
    # Só o preço final: no GBM, dá para sortear S_T direto, sem simular os dias intermediários.
    shocks = rng.standard_normal(number_of_paths)
    return start_price * np.exp((drift - 0.5 * volatility ** 2) * years + volatility * np.sqrt(years) * shocks)


rng = np.random.default_rng(SETTINGS.random_seed)
NEUTRAL_DRIFT = RISK_FREE_RATE - DIVIDEND_YIELD
trading_steps = max(int(round(T_YEARS * TRADING_DAYS_PER_YEAR)), 1)
paths = simulate_gbm_paths(SPOT, PRICE_DRIFT, RECENT_VOLATILITY, T_YEARS, trading_steps, 40, rng)
real_world_terminal = simulate_terminal_prices(SPOT, PRICE_DRIFT, RECENT_VOLATILITY, T_YEARS, 200_000, rng)
neutral_terminal = simulate_terminal_prices(SPOT, NEUTRAL_DRIFT, RECENT_VOLATILITY, T_YEARS, 200_000, rng)

figure, (paths_axis, terminal_axis) = plt.subplots(1, 2, figsize=(14, 4.2))
paths_axis.plot(np.arange(trading_steps + 1), paths.T, color=COLORS["stock"], lw=0.8, alpha=0.5)
paths_axis.axhline(EXAMPLE_PUT.strike, color=COLORS["loss"], lw=1, ls="--")
paths_axis.annotate("strike", (0, EXAMPLE_PUT.strike), xytext=(2, -12), textcoords="offset points", color=COLORS["loss"])
paths_axis.set_title(f"40 trajetórias do GBM (σ = {RECENT_VOLATILITY:.0%}, μ histórico = {PRICE_DRIFT:.0%})")
paths_axis.set_xlabel("pregões até o vencimento"); paths_axis.set_ylabel("preço")
bins = np.linspace(*np.percentile(real_world_terminal, [0.2, 99.8]), 120)
terminal_axis.hist(real_world_terminal, bins=bins, density=True, alpha=0.5, color=COLORS["stock"], label="mundo real (μ histórico)")
terminal_axis.hist(neutral_terminal, bins=bins, density=True, alpha=0.5, color=COLORS["protected"], label="neutro ao risco (r − q)")
terminal_axis.axvline(EXAMPLE_PUT.strike, color=COLORS["loss"], lw=1, ls="--")
terminal_axis.set_yticks([]); terminal_axis.set_title("Distribuição do preço no vencimento"); terminal_axis.legend()
plt.tight_layout(); plt.show()

report("o que a simulação mostra", [
    f"Probabilidade de terminar abaixo da strike: **{(real_world_terminal < EXAMPLE_PUT.strike).mean():.1%}** no mundo real "
    f"(μ = {PRICE_DRIFT:.1%}) e **{(neutral_terminal < EXAMPLE_PUT.strike).mean():.1%}** no mundo neutro ao risco "
    f"(μ = r − q = {NEUTRAL_DRIFT:.1%}).",
    "As duas distribuições têm a **mesma largura** (mesmo σ) e só mudam de posição. Na fórmula do BS só entra a versão "
    "neutra ao risco: a seção A4 mostra que o μ real não afeta o preço justo da put.",
    "O GBM simulado não tem caudas gordas nem agrupamento de volatilidade. Compare com os gráficos anteriores: é por isso "
    "que o mercado precifica as puts de strike baixo acima do que o BS com σ único sugere.",
])

> **Para julgar artigos.** Desconfie de artigos que simulam o mercado com GBM e depois afirmam que um agente "aprendeu a proteger": num GBM não há caudas gordas nem mudança de regime, então o ambiente é mais fácil que o mundo real. Procure testes com **dados históricos** ou com geradores que reproduzam caudas e agrupamento de volatilidade (GARCH, bootstrap de blocos).

## A3. A fórmula da put, peça por peça

**O que vamos fazer.** Implementar a fórmula de Black e Scholes (1973), com o ajuste de Merton (1973) para dividendos contínuos $q$:

$$d_1 = \frac{\ln(S/K) + (r - q + \sigma^2/2)\,T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

$$P = K e^{-rT}\,N(-d_2) \;-\; S e^{-qT}\,N(-d_1)$$

$N(\cdot)$ é a função de distribuição acumulada da normal padrão. Como ler os dois termos:
- $N(-d_2)$ é a **probabilidade, no mundo neutro ao risco, de a put ser exercida** ($S_T < K$).
- $K e^{-rT} N(-d_2)$ é o **valor presente da strike que você recebe**, ponderado pela chance de receber.
- $S e^{-qT} N(-d_1)$ é o **valor presente da ação que você entrega** ao exercer, ponderado da mesma forma (com outra probabilidade, porque o valor da ação entregue depende de quanto ela vale nos cenários de exercício).

A put vale a diferença: o que você recebe menos o que entrega, nos cenários em que exerce.

Primeiro usamos a **volatilidade histórica dos últimos 12 meses**, o "chute" mais simples para σ. Na seção A7 vamos inverter a conta para descobrir que σ o mercado está usando.

In [ ]:
def d1_d2(stock_price, strike, years, rate, dividend_yield, volatility) -> tuple:
    stock_price = np.asarray(stock_price, float)
    volatility_times_root_time = volatility * np.sqrt(years)
    d1 = (np.log(stock_price / strike) + (rate - dividend_yield + 0.5 * volatility ** 2) * years) / volatility_times_root_time
    return d1, d1 - volatility_times_root_time


def bs_put_price(stock_price, strike, years, rate, dividend_yield, volatility):
    d1, d2 = d1_d2(stock_price, strike, years, rate, dividend_yield, volatility)
    return (strike * np.exp(-rate * years) * stats.norm.cdf(-d2)
            - stock_price * np.exp(-dividend_yield * years) * stats.norm.cdf(-d1))


def bs_call_price(stock_price, strike, years, rate, dividend_yield, volatility):
    d1, d2 = d1_d2(stock_price, strike, years, rate, dividend_yield, volatility)
    return (stock_price * np.exp(-dividend_yield * years) * stats.norm.cdf(d1)
            - strike * np.exp(-rate * years) * stats.norm.cdf(d2))


def explain_put_price(stock_price, strike, years, rate, dividend_yield, volatility) -> pd.DataFrame:
    # Cada passo da fórmula, com o valor e o significado.
    d1, d2 = d1_d2(stock_price, strike, years, rate, dividend_yield, volatility)
    discounted_strike = strike * np.exp(-rate * years)
    discounted_stock = stock_price * np.exp(-dividend_yield * years)
    rows = [
        ("S (preço da ação)", stock_price, "dado do mercado"),
        ("K (strike)", strike, "contrato"),
        ("T (anos até o vencimento)", years, "dias corridos ÷ 365"),
        ("r (juros ao ano)", rate, "^IRX (Treasury de 13 semanas)"),
        ("q (dividend yield)", dividend_yield, "dividendos dos últimos 12 meses ÷ preço"),
        ("σ (volatilidade ao ano)", volatility, "aqui: histórica de 12 meses"),
        ("d1", float(d1), "distância padronizada entre preço e strike, com ajuste de drift e volatilidade"),
        ("d2 = d1 − σ√T", float(d2), "a mesma distância, sem o ajuste de σ²"),
        ("N(−d2)", float(stats.norm.cdf(-d2)), "probabilidade neutra ao risco de exercer a put"),
        ("N(−d1)", float(stats.norm.cdf(-d1)), "peso da ação entregue (também é −delta ÷ e^(−qT))"),
        ("K·e^(−rT)", discounted_strike, "valor presente da strike"),
        ("termo 1: K·e^(−rT)·N(−d2)", discounted_strike * float(stats.norm.cdf(-d2)), "o que você espera receber, a valor presente"),
        ("termo 2: S·e^(−qT)·N(−d1)", discounted_stock * float(stats.norm.cdf(-d1)), "o valor da ação que você espera entregar"),
        ("preço da put = termo 1 − termo 2", float(bs_put_price(stock_price, strike, years, rate, dividend_yield, volatility)), "Black-Scholes"),
    ]
    return pd.DataFrame(rows, columns=["passo", "valor", "o que significa"]).set_index("passo")


EXAMPLE_INPUTS = dict(stock_price=SPOT, strike=EXAMPLE_PUT.strike, years=T_YEARS, rate=RISK_FREE_RATE,
                      dividend_yield=DIVIDEND_YIELD, volatility=RECENT_VOLATILITY)
explanation = explain_put_price(**EXAMPLE_INPUTS)
display(explanation.style.format({"valor": "{:,.4f}"}))
BS_PRICE_HISTORICAL_VOL = explanation.loc["preço da put = termo 1 − termo 2", "valor"]

In [ ]:
def price_sensitivity_table(inputs: dict) -> pd.DataFrame:
    # Muda um insumo de cada vez e mostra o efeito no preço da put.
    base_price = float(bs_put_price(**inputs))
    scenarios = {
        "ação cai 5%": {"stock_price": inputs["stock_price"] * 0.95},
        "ação sobe 5%": {"stock_price": inputs["stock_price"] * 1.05},
        "strike 5% maior": {"strike": inputs["strike"] * 1.05},
        "prazo dobra": {"years": inputs["years"] * 2},
        "volatilidade +5 p.p.": {"volatility": inputs["volatility"] + 0.05},
        "juros +1 p.p.": {"rate": inputs["rate"] + 0.01},
        "dividendos +1 p.p.": {"dividend_yield": inputs["dividend_yield"] + 0.01},
    }
    rows = []
    for name, change in scenarios.items():
        new_price = float(bs_put_price(**{**inputs, **change}))
        rows.append({"cenário": name, "preço da put": new_price, "variação": new_price / base_price - 1})
    return pd.DataFrame(rows).set_index("cenário")


SENSITIVITY = price_sensitivity_table(EXAMPLE_INPUTS)
display(SENSITIVITY.style.format({"preço da put": "{:,.3f}", "variação": "{:+.1%}"}))

exercise_probability = explanation.loc["N(−d2)", "valor"]
report("a fórmula com números reais", [
    f"Com σ histórico de {RECENT_VOLATILITY:.1%}, o Black-Scholes diz que a put vale **{BS_PRICE_HISTORICAL_VOL:,.3f}**. "
    f"O mercado cobra **{EXAMPLE_PREMIUM:,.3f}** (ask) e o mid é {EXAMPLE_MID:,.3f}: o mercado está "
    f"{EXAMPLE_MID / BS_PRICE_HISTORICAL_VOL - 1:+.0%} em relação ao BS com volatilidade histórica.",
    "Essa diferença **não é um erro de conta**: ela diz que o mercado usa um σ diferente do histórico. "
    "Descobrir esse σ é a volatilidade implícita (A7).",
    f"A probabilidade neutra ao risco de exercício, N(−d2), é **{exercise_probability:.1%}**. Ela **não é** a "
    "probabilidade real de a ação cair abaixo da strike: é a probabilidade num mundo em que todos os ativos rendem os juros.",
    f"Na tabela de sensibilidade, o insumo que mais mexe no preço é **{SENSITIVITY['variação'].abs().idxmax()}** "
    f"({SENSITIVITY['variação'].abs().max():+.0%}). Puts ficam mais caras com ação em queda, strike maior, prazo maior "
    "e volatilidade maior; ficam mais baratas com juros maiores (o valor presente da strike cai).",
])

> **Para julgar artigos.** Confira **que σ** o artigo pôs na fórmula. Usar volatilidade histórica para "prever" o preço da opção e depois medir o "erro do BS" mistura duas coisas: o erro do modelo e o erro da previsão de volatilidade. Artigos sólidos comparam com o preço de mercado usando volatilidade implícita, ou separam explicitamente os dois erros.

## A4. Por que o retorno esperado da ação some da fórmula

**O que vamos fazer.** Repare que $\mu$ não aparece na fórmula. A razão é a **replicação**: quem vende uma put consegue neutralizar o risco comprando ou vendendo $\Delta$ ações (o *delta hedge*) e ajustando essa quantidade com frequência. Se a carteira replica a put em todos os cenários, o preço justo da put é o custo de montar a carteira, e esse custo não depende de a ação "ir subir ou cair".

Vamos verificar isso por simulação, no ponto de vista de quem **vende** a put (como nos tópicos de hedge do curso):
1. Vende a put pelo preço do BS e recebe o prêmio.
2. Fica com $\Delta = -e^{-qT}N(-d_1)$ ações (vendido em ações, porque a put perde valor quando a ação sobe) e aplica o resto a juros.
3. Rebalanceia $n$ vezes até o vencimento.
4. No vencimento, compara a carteira com o que precisa pagar, $\max(K - S_T, 0)$. A diferença é o **erro de hedge**.

Se o BS estiver certo, o erro médio é **zero para qualquer μ**, e o desvio do erro **cai à medida que $n$ aumenta**.

In [ ]:
def bs_put_delta(stock_price, strike, years, rate, dividend_yield, volatility):
    d1, _ = d1_d2(stock_price, strike, years, rate, dividend_yield, volatility)
    return -np.exp(-dividend_yield * years) * stats.norm.cdf(-d1)


def simulate_delta_hedge_error(start_price: float, strike: float, years: float, rate: float, dividend_yield: float,
                               volatility: float, price_drift: float, rebalances: int, number_of_paths: int,
                               rng: np.random.Generator) -> np.ndarray:
    # Vende a put, faz delta hedge com n rebalanceamentos; devolve o erro final a valor presente.
    step_years = years / rebalances
    stock = np.full(number_of_paths, start_price)
    delta = bs_put_delta(stock, strike, years, rate, dividend_yield, volatility)
    cash = bs_put_price(start_price, strike, years, rate, dividend_yield, volatility) - delta * stock
    for step in range(1, rebalances + 1):
        shocks = rng.standard_normal(number_of_paths)
        new_stock = stock * np.exp((price_drift - 0.5 * volatility ** 2) * step_years + volatility * np.sqrt(step_years) * shocks)
        cash = cash * np.exp(rate * step_years) + delta * stock * dividend_yield * step_years   # juros e dividendos
        stock = new_stock
        years_left = years - step * step_years
        if years_left > 1e-10:
            new_delta = bs_put_delta(stock, strike, years_left, rate, dividend_yield, volatility)
        else:
            new_delta = np.where(stock < strike, -1.0, 0.0)
        cash -= (new_delta - delta) * stock
        delta = new_delta
    hedging_error = cash + delta * stock - np.maximum(strike - stock, 0.0)
    return hedging_error * np.exp(-rate * years)


rng = np.random.default_rng(SETTINGS.random_seed)
REBALANCE_COUNTS = [4, 12, 52, 252]
DRIFT_SCENARIOS = {"μ histórico": PRICE_DRIFT, "μ = −30% (mercado em queda)": -0.30, "μ = +30% (mercado em alta)": 0.30}
hedge_inputs = dict(start_price=SPOT, strike=EXAMPLE_PUT.strike, years=T_YEARS, rate=RISK_FREE_RATE,
                    dividend_yield=DIVIDEND_YIELD, volatility=RECENT_VOLATILITY)
reference_price = float(bs_put_price(SPOT, EXAMPLE_PUT.strike, T_YEARS, RISK_FREE_RATE, DIVIDEND_YIELD, RECENT_VOLATILITY))
hedge_rows, stored_errors = [], {}
for drift_name, drift in DRIFT_SCENARIOS.items():
    for rebalances in REBALANCE_COUNTS:
        errors = simulate_delta_hedge_error(**hedge_inputs, price_drift=drift, rebalances=rebalances,
                                            number_of_paths=SETTINGS.hedge_simulation_paths, rng=rng)
        stored_errors[(drift_name, rebalances)] = errors
        hedge_rows.append({"cenário": drift_name, "rebalanceamentos": rebalances,
                           "erro médio (% do prêmio)": errors.mean() / reference_price,
                           "desvio do erro (% do prêmio)": errors.std() / reference_price})
HEDGE_TABLE = pd.DataFrame(hedge_rows)
display(HEDGE_TABLE.pivot(index="rebalanceamentos", columns="cenário", values=["erro médio (% do prêmio)", "desvio do erro (% do prêmio)"])
        .style.format("{:+.1%}"))

figure, axis = plt.subplots(figsize=(10, 3.8))
for rebalances, color in [(12, COLORS["muted"]), (252, COLORS["protected"])]:
    errors = stored_errors[("μ histórico", rebalances)] / reference_price
    axis.hist(errors, bins=80, range=(-1.5, 1.5), alpha=0.55, color=color, label=f"{rebalances} rebalanceamentos")
axis.axvline(0, color=COLORS["ink"], lw=1)
axis.xaxis.set_major_formatter(PercentFormatter(1.0))
axis.set_xlabel("erro de hedge no vencimento (% do prêmio da put)")
axis.set_title("Quanto mais frequente o rebalanceamento, mais o delta hedge replica a put")
axis.set_yticks([]); axis.legend(); plt.show()

std_by_count = HEDGE_TABLE[HEDGE_TABLE["cenário"] == "μ histórico"].set_index("rebalanceamentos")["desvio do erro (% do prêmio)"]
report("replicação e a ausência do μ", [
    "O **erro médio** fica perto de zero nos três cenários de μ (−30%, histórico e +30%). Isso é a prova numérica de que "
    "o preço do BS não depende do retorno esperado da ação: quem vende a put e faz o hedge não ganha nem perde em média.",
    f"O **desvio do erro** cai de {std_by_count.iloc[0]:.0%} do prêmio (4 rebalanceamentos) para {std_by_count.iloc[-1]:.0%} "
    f"(252, um por pregão). A teoria prevê queda proporcional a 1/√n: de 4 para 252 rebalanceamentos, o fator teórico é "
    f"{np.sqrt(252 / 4):.1f}× e o observado é {std_by_count.iloc[0] / std_by_count.iloc[-1]:.1f}×.",
    "Esse resultado só vale **dentro do GBM**. No mundo real há caudas gordas, saltos e custos de transação, e o hedge "
    "nunca é perfeito. É esse buraco que os trabalhos de *deep hedging* (Buehler et al., 2019) tentam preencher com IA.",
    "Para o seu mestrado: você está do lado **comprado** (compra a put e não faz hedge dinâmico). O que importa para você "
    "é se o preço cobrado é justo em relação ao risco que você corre, o que é a pergunta da seção A9.",
])

> **Para julgar artigos.** Em artigos de hedge com aprendizado por reforço, verifique: (1) se o ambiente simulado é GBM (onde o delta do BS já é ótimo sem custos, então "bater o BS" exige custos de transação ou outro modelo); (2) se a comparação é contra o **delta hedge** com a mesma frequência de rebalanceamento; (3) se reportam a **distribuição** do erro (desvio, cauda), não só a média.

## A5. Prova por Monte Carlo: a fórmula é a média dos payoffs descontados

**O que vamos fazer.** No mundo neutro ao risco, o preço de qualquer opção é o valor esperado do payoff, trazido a valor presente:

$$P = e^{-rT}\ \mathbb{E}^{\mathbb{Q}}\big[\max(K - S_T, 0)\big]$$

Vamos sortear $S_T$ com drift $r - q$, calcular a média dos payoffs descontados e ver o resultado convergir para a fórmula fechada à medida que aumentamos o número de cenários. O erro-padrão da média cai com $1/\sqrt{N}$.

In [ ]:
def monte_carlo_put_price(stock_price, strike, years, rate, dividend_yield, volatility, number_of_paths, rng) -> tuple:
    terminal_prices = simulate_terminal_prices(stock_price, rate - dividend_yield, volatility, years, number_of_paths, rng)
    discounted_payoffs = np.exp(-rate * years) * np.maximum(strike - terminal_prices, 0.0)
    return discounted_payoffs.mean(), discounted_payoffs.std() / np.sqrt(number_of_paths)


rng = np.random.default_rng(SETTINGS.random_seed)
closed_form_price = float(bs_put_price(**EXAMPLE_INPUTS))
convergence_rows = []
for number_of_paths in [100, 1_000, 10_000, 100_000, 1_000_000]:
    estimate, standard_error = monte_carlo_put_price(**EXAMPLE_INPUTS, number_of_paths=number_of_paths, rng=rng)
    convergence_rows.append({"cenários": number_of_paths, "preço Monte Carlo": estimate, "erro-padrão": standard_error,
                             "IC 95% inferior": estimate - 1.96 * standard_error, "IC 95% superior": estimate + 1.96 * standard_error,
                             "fórmula fechada": closed_form_price, "diferença": estimate - closed_form_price})
CONVERGENCE = pd.DataFrame(convergence_rows).set_index("cenários")
display(CONVERGENCE.style.format("{:,.4f}"))

figure, axis = plt.subplots(figsize=(10, 3.6))
axis.fill_between(CONVERGENCE.index, CONVERGENCE["IC 95% inferior"], CONVERGENCE["IC 95% superior"], color=COLORS["stock"], alpha=0.2)
axis.plot(CONVERGENCE.index, CONVERGENCE["preço Monte Carlo"], "o-", color=COLORS["stock"], label="Monte Carlo (IC 95%)")
axis.axhline(closed_form_price, color=COLORS["protected"], ls="--", label="fórmula fechada")
axis.set_xscale("log"); axis.set_xlabel("número de cenários"); axis.set_ylabel("preço da put")
axis.set_title("Monte Carlo neutro ao risco converge para Black-Scholes"); axis.legend(); plt.show()

covered = ((CONVERGENCE["IC 95% inferior"] <= closed_form_price) & (closed_form_price <= CONVERGENCE["IC 95% superior"])).sum()
report("Monte Carlo × fórmula", [
    f"Com 1 milhão de cenários, o Monte Carlo dá **{CONVERGENCE['preço Monte Carlo'].iloc[-1]:,.4f}** e a fórmula "
    f"**{closed_form_price:,.4f}** (diferença de {CONVERGENCE['diferença'].iloc[-1]:+.4f}).",
    f"O intervalo de 95% do Monte Carlo contém a fórmula em **{covered} de {len(CONVERGENCE)}** tamanhos de amostra.",
    f"O erro-padrão cai de {CONVERGENCE['erro-padrão'].iloc[0]:.3f} (100 cenários) para {CONVERGENCE['erro-padrão'].iloc[-1]:.4f} "
    "(1 milhão): 100 vezes menos com 10 mil vezes mais cenários, como manda a regra de 1/√N.",
    "Por que isso importa: para opções sem fórmula fechada (americanas, exóticas, modelos com caudas gordas), o preço "
    "é calculado exatamente assim, por simulação. A fórmula do BS é só o caso em que a integral tem solução exata.",
])

> **Para julgar artigos.** Todo resultado de Monte Carlo deve vir com o **número de cenários** e o **erro-padrão** (ou intervalo de confiança). Diferenças entre métodos menores que dois erros-padrão não são diferenças.

## A6. As gregas e a relação Delta × preço

**O que vamos fazer.** As gregas são as derivadas do preço da put em relação a cada insumo. Elas dizem **quanto a put muda** quando algo muda:

| Grega | Fórmula (put, com dividendos) | Significado prático |
|---|---|---|
| Delta $\Delta$ | $-e^{-qT}N(-d_1)$ | variação da put para cada US$ 1 na ação; entre −1 e 0 |
| Gama $\Gamma$ | $\dfrac{e^{-qT}\varphi(d_1)}{S\sigma\sqrt{T}}$ | quanto o delta muda para cada US$ 1 na ação |
| Vega $\mathcal{V}$ | $S e^{-qT}\varphi(d_1)\sqrt{T}$ | variação da put para cada 1,00 (100 p.p.) de volatilidade |
| Teta $\Theta$ | $-\dfrac{S e^{-qT}\varphi(d_1)\sigma}{2\sqrt{T}} + rKe^{-rT}N(-d_2) - qSe^{-qT}N(-d_1)$ | variação da put por ano que passa (dividimos por 365 para "por dia") |
| Rô $\rho$ | $-KTe^{-rT}N(-d_2)$ | variação da put para cada 1,00 de juros |

$\varphi$ é a densidade da normal padrão. Para a **posição protegida** (1 ação + 1 put), o delta total é $1 + \Delta_{put}$: é a fração da exposição à ação que **continua desprotegida** no curto prazo.

Primeiro conferimos cada fórmula contra **diferenças finitas** (mudar o insumo um pouquinho para cima e para baixo e medir a variação do preço). Se a fórmula estiver certa, os dois números coincidem.

In [ ]:
def bs_gamma(stock_price, strike, years, rate, dividend_yield, volatility):
    d1, _ = d1_d2(stock_price, strike, years, rate, dividend_yield, volatility)
    return np.exp(-dividend_yield * years) * stats.norm.pdf(d1) / (np.asarray(stock_price) * volatility * np.sqrt(years))


def bs_vega(stock_price, strike, years, rate, dividend_yield, volatility):
    d1, _ = d1_d2(stock_price, strike, years, rate, dividend_yield, volatility)
    return np.asarray(stock_price) * np.exp(-dividend_yield * years) * stats.norm.pdf(d1) * np.sqrt(years)


def bs_put_theta(stock_price, strike, years, rate, dividend_yield, volatility):
    d1, d2 = d1_d2(stock_price, strike, years, rate, dividend_yield, volatility)
    stock_price = np.asarray(stock_price)
    return (-stock_price * np.exp(-dividend_yield * years) * stats.norm.pdf(d1) * volatility / (2 * np.sqrt(years))
            + rate * strike * np.exp(-rate * years) * stats.norm.cdf(-d2)
            - dividend_yield * stock_price * np.exp(-dividend_yield * years) * stats.norm.cdf(-d1))


def bs_put_rho(stock_price, strike, years, rate, dividend_yield, volatility):
    _, d2 = d1_d2(stock_price, strike, years, rate, dividend_yield, volatility)
    return -strike * years * np.exp(-rate * years) * stats.norm.cdf(-d2)


def finite_difference_greeks(inputs: dict) -> dict:
    # Derivadas numéricas centrais: (P(x + h) − P(x − h)) ÷ 2h.
    def price_with(**change):
        return float(bs_put_price(**{**inputs, **change}))
    S, sigma, T, r = inputs["stock_price"], inputs["volatility"], inputs["years"], inputs["rate"]
    h_stock, h_vol, h_time, h_rate = S * 1e-4, 1e-4, 1e-5, 1e-5
    base = price_with()
    return {
        "delta": (price_with(stock_price=S + h_stock) - price_with(stock_price=S - h_stock)) / (2 * h_stock),
        "gama": (price_with(stock_price=S + h_stock) - 2 * base + price_with(stock_price=S - h_stock)) / h_stock ** 2,
        "vega": (price_with(volatility=sigma + h_vol) - price_with(volatility=sigma - h_vol)) / (2 * h_vol),
        "teta (por ano)": -(price_with(years=T + h_time) - price_with(years=T - h_time)) / (2 * h_time),
        "rô": (price_with(rate=r + h_rate) - price_with(rate=r - h_rate)) / (2 * h_rate),
    }


analytic_greeks = {"delta": bs_put_delta(**EXAMPLE_INPUTS), "gama": bs_gamma(**EXAMPLE_INPUTS),
                   "vega": bs_vega(**EXAMPLE_INPUTS), "teta (por ano)": bs_put_theta(**EXAMPLE_INPUTS),
                   "rô": bs_put_rho(**EXAMPLE_INPUTS)}
numeric_greeks = finite_difference_greeks(EXAMPLE_INPUTS)
GREEKS_CHECK = pd.DataFrame({"fórmula analítica": {k: float(v) for k, v in analytic_greeks.items()},
                             "diferenças finitas": numeric_greeks})
GREEKS_CHECK["erro relativo"] = (GREEKS_CHECK["fórmula analítica"] / GREEKS_CHECK["diferenças finitas"] - 1).abs()
display(GREEKS_CHECK.style.format({"fórmula analítica": "{:,.6f}", "diferenças finitas": "{:,.6f}", "erro relativo": "{:.1e}"}))

In [ ]:
stock_grid = np.linspace(SPOT * 0.7, SPOT * 1.3, 300)
maturity_choices = {"1/4 do prazo": T_YEARS / 4, "prazo da put": T_YEARS, "4× o prazo": T_YEARS * 4}
common = dict(strike=EXAMPLE_PUT.strike, rate=RISK_FREE_RATE, dividend_yield=DIVIDEND_YIELD, volatility=RECENT_VOLATILITY)
line_colors = [COLORS["muted"], COLORS["protected"], COLORS["stock"]]

figure, axes = plt.subplots(2, 2, figsize=(13, 7.5))
for (label, years), color in zip(maturity_choices.items(), line_colors):
    axes[0, 0].plot(stock_grid, bs_put_delta(stock_grid, years=years, **common), color=color, label=label)
    axes[0, 1].plot(stock_grid, bs_gamma(stock_grid, years=years, **common), color=color, label=label)
    axes[1, 0].plot(stock_grid, bs_vega(stock_grid, years=years, **common) / 100, color=color, label=label)
    axes[1, 1].plot(stock_grid, 1 + bs_put_delta(stock_grid, years=years, **common), color=color, label=label)
titles = ["Delta da put × preço da ação", "Gama × preço da ação",
          "Vega por 1 p.p. de volatilidade × preço", "Delta da posição protegida (1 + Δ put)"]
for axis, title in zip(axes.ravel(), titles):
    axis.axvline(EXAMPLE_PUT.strike, color=COLORS["loss"], lw=1, ls="--")
    axis.axvline(SPOT, color=COLORS["ink"], lw=1, ls=":")
    axis.set_title(title); axis.set_xlabel("preço da ação")
axes[0, 0].legend(); axes[1, 1].yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout(); plt.show()

example_delta = float(analytic_greeks["delta"])
report("o que as gregas dizem sobre a sua proteção", [
    f"As fórmulas analíticas batem com as diferenças finitas (maior erro relativo: {GREEKS_CHECK['erro relativo'].max():.0e}). "
    "Isso confirma que o código implementa as derivadas corretas.",
    f"**Delta = {example_delta:+.3f}**: se a ação cair US$ 1, a put ganha cerca de US$ {abs(example_delta):.2f}. "
    f"A posição protegida tem delta **{1 + example_delta:.2f}**, ou seja, hoje ela ainda sente {1 + example_delta:.0%} "
    "das quedas pequenas. A proteção \"liga\" de verdade quando a ação se aproxima da strike (o delta tende a −1).",
    f"**Vega = {float(analytic_greeks['vega']) / 100:,.3f} por p.p.**: se a volatilidade implícita subir 1 ponto percentual, "
    f"a put fica US$ {float(analytic_greeks['vega']) / 100:,.3f} mais cara. Para quem **compra** proteção, vega é o "
    "risco de entrar quando o seguro está caro (volatilidade alta).",
    f"**Teta = {float(analytic_greeks['teta (por ano)']) / 365:,.4f} por dia**: é o quanto a put perde de valor a cada dia "
    "parado, se nada mais mudar. É o \"aluguel\" diário do seguro.",
    "No gráfico de gama, o pico fica perto da strike e fica mais alto para prazos curtos: é onde o delta muda mais rápido. "
    "No gráfico de vega, prazos longos têm vega maior: puts longas são mais sensíveis a mudanças de volatilidade.",
])

> **Para julgar artigos.** Gregas calculadas pelo BS assumem σ constante. Num mercado com smile, o "delta verdadeiro" é diferente (o σ muda quando o preço muda). Artigos de hedge que usam o delta do BS como referência deveriam dizer **qual σ** usaram (implícito da strike, ATM ou histórico).

## A7. Volatilidade implícita, smile e o exercício antecipado

**O que vamos fazer.** Inverter a fórmula: dado o preço de mercado da put, encontrar o $\sigma$ que faz o BS reproduzir esse preço. É a **volatilidade implícita (VI)**. Como o preço da put cresce sempre com σ (vega > 0), existe no máximo uma solução, que encontramos com o método de Brent.

Dois cuidados com dados reais:
1. **Exercício americano.** Opções de ações dos EUA podem ser exercidas antes do vencimento. Isso vale um pouco mais que a versão europeia, sobretudo para puts dentro do dinheiro (strike acima do preço). Calculamos o preço americano com a **árvore binomial de Cox, Ross e Rubinstein (1979)**: o tempo é dividido em passos, em cada passo a ação sobe por $u = e^{\sigma\sqrt{\Delta t}}$ ou cai por $d = 1/u$, com probabilidade neutra ao risco $p = \frac{e^{(r-q)\Delta t} - d}{u - d}$; de trás para frente, em cada nó o valor é o **maior** entre exercer agora e esperar.
2. **Qual preço inverter.** Usamos o **mid** para a VI de referência e o **ask** para a VI que você efetivamente paga.

Antes de usar a árvore, conferimos que a versão europeia dela converge para o BS.

In [ ]:
def binomial_put_price(stock_price, strike, years, rate, dividend_yield, volatility, steps, american=True) -> float:
    # Árvore de Cox-Ross-Rubinstein. american=True permite exercer antes do vencimento.
    step_years = years / steps
    up = np.exp(volatility * np.sqrt(step_years)); down = 1 / up
    up_probability = (np.exp((rate - dividend_yield) * step_years) - down) / (up - down)
    discount = np.exp(-rate * step_years)
    up_moves = np.arange(steps + 1)
    values = np.maximum(strike - stock_price * up ** up_moves * down ** (steps - up_moves), 0.0)
    for step in range(steps - 1, -1, -1):
        values = discount * (up_probability * values[1:] + (1 - up_probability) * values[:-1])
        if american:
            up_moves = np.arange(step + 1)
            node_prices = stock_price * up ** up_moves * down ** (step - up_moves)
            values = np.maximum(values, strike - node_prices)
    return float(values[0])


def implied_volatility(option_price: float, pricing_function, low: float = 1e-4, high: float = 5.0) -> float:
    # σ que faz pricing_function(σ) = preço observado; NaN se o preço estiver fora do intervalo possível.
    if not np.isfinite(option_price) or option_price <= 0:
        return np.nan
    def gap(volatility):
        return pricing_function(volatility) - option_price
    if gap(low) > 0 or gap(high) < 0:
        return np.nan
    return optimize.brentq(gap, low, high, xtol=1e-6)


tree_check = pd.DataFrame([{"passos": steps,
                            "árvore europeia": binomial_put_price(**EXAMPLE_INPUTS, steps=steps, american=False),
                            "Black-Scholes": closed_form_price,
                            "árvore americana": binomial_put_price(**EXAMPLE_INPUTS, steps=steps, american=True)}
                           for steps in [25, 50, 100, 300, 1000]]).set_index("passos")
tree_check["erro da árvore europeia"] = tree_check["árvore europeia"] - tree_check["Black-Scholes"]
display(tree_check.style.format("{:,.5f}"))

In [ ]:
def add_implied_volatilities(puts: pd.DataFrame, stock_price: float, years: float, rate: float, dividend_yield: float,
                             tree_steps: int) -> pd.DataFrame:
    # VI europeia (BS) e americana (árvore) a partir do mid, e VI europeia a partir do ask.
    # Prêmio do exercício antecipado = árvore americana − árvore europeia (mesma árvore: o erro de discretização se cancela).
    puts = puts.copy()
    def european_pricer(strike):
        return lambda volatility: float(bs_put_price(stock_price, strike, years, rate, dividend_yield, volatility))
    def american_pricer(strike):
        return lambda volatility: binomial_put_price(stock_price, strike, years, rate, dividend_yield, volatility, tree_steps)
    puts["moneyness"] = puts["strike"] / stock_price
    puts["vi_bs_mid"] = [implied_volatility(row.mid, european_pricer(row.strike)) for row in puts.itertuples()]
    puts["vi_americana_mid"] = [implied_volatility(row.mid, american_pricer(row.strike)) for row in puts.itertuples()]
    puts["vi_bs_ask"] = [implied_volatility(row.purchase_price, european_pricer(row.strike)) for row in puts.itertuples()]
    puts["premio_exercicio_antecipado"] = [
        binomial_put_price(stock_price, row.strike, years, rate, dividend_yield, row.vi_americana_mid, tree_steps)
        - binomial_put_price(stock_price, row.strike, years, rate, dividend_yield, row.vi_americana_mid, tree_steps, american=False)
        if np.isfinite(row.vi_americana_mid) else np.nan for row in puts.itertuples()]
    return puts


PUTS = add_implied_volatilities(PUTS, SPOT, T_YEARS, RISK_FREE_RATE, DIVIDEND_YIELD, SETTINGS.binomial_steps_for_iv)
smile_columns = ["strike", "moneyness", "mid", "purchase_price", "vi_bs_mid", "vi_americana_mid", "vi_bs_ask",
                 "impliedVolatility", "premio_exercicio_antecipado"]
display(PUTS[smile_columns].rename(columns={"purchase_price": "ask (ou último)", "impliedVolatility": "vi_yahoo"})
        .style.format({"strike": "{:,.2f}", "moneyness": "{:.0%}", "mid": "{:,.3f}", "ask (ou último)": "{:,.3f}",
                       "vi_bs_mid": "{:.1%}", "vi_americana_mid": "{:.1%}", "vi_bs_ask": "{:.1%}", "vi_yahoo": "{:.1%}",
                       "premio_exercicio_antecipado": "{:,.4f}"}))

figure, (smile_axis, premium_axis) = plt.subplots(1, 2, figsize=(14, 4.2))
smile_axis.plot(PUTS["moneyness"], PUTS["vi_americana_mid"], "o-", color=COLORS["protected"], label="VI americana (mid)")
smile_axis.plot(PUTS["moneyness"], PUTS["vi_bs_mid"], "o-", color=COLORS["stock"], ms=4, lw=1, label="VI europeia/BS (mid)")
smile_axis.plot(PUTS["moneyness"], PUTS["vi_bs_ask"], "--", color=COLORS["muted"], lw=1, label="VI europeia (ask)")
smile_axis.plot(PUTS["moneyness"], PUTS["impliedVolatility"], ":", color=COLORS["ink"], lw=1, label="VI do Yahoo")
smile_axis.axhline(RECENT_VOLATILITY, color=COLORS["accent"], lw=1, ls="--", label="vol. histórica 12m")
smile_axis.axvline(1, color=COLORS["ink"], lw=1, ls=":")
smile_axis.xaxis.set_major_formatter(PercentFormatter(1.0)); smile_axis.yaxis.set_major_formatter(PercentFormatter(1.0))
smile_axis.set_xlabel("strike ÷ preço da ação"); smile_axis.set_title("Smile: a volatilidade que o mercado cobra em cada strike")
smile_axis.legend(fontsize=8)
premium_axis.bar(PUTS["moneyness"], PUTS["premio_exercicio_antecipado"] / PUTS["mid"], width=0.006, color=COLORS["purple"])
premium_axis.xaxis.set_major_formatter(PercentFormatter(1.0)); premium_axis.yaxis.set_major_formatter(PercentFormatter(1.0))
premium_axis.set_xlabel("strike ÷ preço da ação"); premium_axis.set_title("Prêmio do exercício antecipado (% do preço da put)")
plt.tight_layout(); plt.show()


def interpolate_smile(puts: pd.DataFrame, moneyness: float, column: str = "vi_americana_mid") -> float:
    valid = puts[["moneyness", column]].dropna().sort_values("moneyness")
    return float(np.interp(moneyness, valid["moneyness"], valid[column])) if len(valid) >= 2 else np.nan


atm_vol, otm_vol = interpolate_smile(PUTS, 1.0), interpolate_smile(PUTS, 0.90)
EXAMPLE_IV = interpolate_smile(PUTS, EXAMPLE_PUT.strike / SPOT)
yahoo_gap = (PUTS["impliedVolatility"] - PUTS["vi_bs_mid"]).abs().median()
report("o que o mercado está cobrando", [
    f"A árvore europeia converge para o BS (erro de {tree_check['erro da árvore europeia'].iloc[-1]:+.5f} com 1000 passos), "
    "então a árvore americana é confiável.",
    f"VI americana no dinheiro (strike = preço): **{atm_vol:.1%}**; a 90% do preço: **{otm_vol:.1%}**. "
    f"Inclinação do smile: **{(otm_vol - atm_vol) * 100:+.1f} p.p.** Puts de strike baixo custam mais volatilidade porque "
    "o mercado cobra pelas quedas fortes que o BS (normal) subestima, as mesmas caudas gordas vistas em A2.",
    f"Comparando com a volatilidade histórica de {RECENT_VOLATILITY:.1%}: a put do exemplo (strike {EXAMPLE_PUT.strike:,.2f}) "
    f"é negociada a VI de **{EXAMPLE_IV:.1%}**.",
    f"O prêmio do exercício antecipado é pequeno fora do dinheiro e cresce dentro do dinheiro (máximo de "
    f"{(PUTS['premio_exercicio_antecipado'] / PUTS['mid']).max():.1%} do preço da put). Para puts de proteção fora do "
    "dinheiro, usar o BS europeu é uma aproximação razoável.",
    f"A VI publicada pelo Yahoo difere da nossa em {yahoo_gap * 100:.1f} p.p. (mediana). O método do Yahoo não é "
    "documentado; por isso o notebook calcula a própria VI, de forma reproduzível.",
    "Entre a VI do mid e a do ask existe o spread: a VI que **você paga** é a do ask.",
])

> **Para julgar artigos.** Verifique (1) se a VI foi calculada com modelo **europeu ou americano** e se isso é compatível com as opções usadas; (2) se veio do **mid, do ask ou do fechamento**; (3) qual taxa de juros e dividendos foram usados na inversão. Pequenas diferenças nesses insumos mudam a VI em vários pontos percentuais, especialmente em opções de prazo curto.

## A8. Paridade put-call: os preços reais são coerentes?

**O que vamos fazer.** Para opções **europeias** de mesma strike e vencimento existe uma identidade exata, que não depende de modelo:

$$C - P = S e^{-qT} - K e^{-rT}$$

Para opções **americanas** ela vira um intervalo (Hull, cap. 11):

$$S - D - K \;\le\; C - P \;\le\; S - K e^{-rT}$$

onde $D$ é o valor presente dos dividendos até o vencimento (aqui, $D \approx S(1 - e^{-qT})$). Se os preços reais respeitam esse intervalo, os dados são coerentes e não há arbitragem óbvia. Se não respeitam, o problema costuma ser **cotação velha** ou spread largo.

In [ ]:
if CALLS is None:
    display(Markdown("> Sem calls líquidas para este vencimento: a paridade não pode ser verificada."))
    PARITY = None
else:
    PARITY = CALLS[["strike", "mid", "bid", "ask"]].merge(PUTS[["strike", "mid", "bid", "ask"]], on="strike", suffixes=("_call", "_put"))
    dividends_present_value = SPOT * (1 - np.exp(-DIVIDEND_YIELD * T_YEARS))
    PARITY["C − P (mid)"] = PARITY["mid_call"] - PARITY["mid_put"]
    PARITY["europeia: S·e^(−qT) − K·e^(−rT)"] = SPOT * np.exp(-DIVIDEND_YIELD * T_YEARS) - PARITY["strike"] * np.exp(-RISK_FREE_RATE * T_YEARS)
    PARITY["limite inferior americano"] = SPOT - dividends_present_value - PARITY["strike"]
    PARITY["limite superior americano"] = SPOT - PARITY["strike"] * np.exp(-RISK_FREE_RATE * T_YEARS)
    PARITY["resíduo europeu"] = PARITY["C − P (mid)"] - PARITY["europeia: S·e^(−qT) − K·e^(−rT)"]
    PARITY["dentro dos limites (mid)"] = PARITY["C − P (mid)"].between(PARITY["limite inferior americano"] - 1e-9,
                                                                       PARITY["limite superior americano"] + 1e-9)
    # Considerando o spread: existe algum preço executável dentro dos limites?
    PARITY["dentro com spread"] = ((PARITY["ask_call"] - PARITY["bid_put"] >= PARITY["limite inferior americano"])
                                   & (PARITY["bid_call"] - PARITY["ask_put"] <= PARITY["limite superior americano"]))
    display(PARITY[["strike", "C − P (mid)", "europeia: S·e^(−qT) − K·e^(−rT)", "limite inferior americano",
                    "limite superior americano", "resíduo europeu", "dentro dos limites (mid)", "dentro com spread"]]
            .style.format({c: "{:,.3f}" for c in ["strike", "C − P (mid)", "europeia: S·e^(−qT) − K·e^(−rT)",
                                                  "limite inferior americano", "limite superior americano", "resíduo europeu"]}))
    figure, axis = plt.subplots(figsize=(10, 3.8))
    axis.fill_between(PARITY["strike"], PARITY["limite inferior americano"] - PARITY["europeia: S·e^(−qT) − K·e^(−rT)"],
                      PARITY["limite superior americano"] - PARITY["europeia: S·e^(−qT) − K·e^(−rT)"],
                      color=COLORS["grid"], label="faixa permitida para opções americanas")
    axis.plot(PARITY["strike"], PARITY["resíduo europeu"], "o-", color=COLORS["protected"], label="(C − P) observado − paridade europeia")
    axis.axhline(0, color=COLORS["ink"], lw=1)
    axis.set_xlabel("strike"); axis.set_ylabel("US$ por ação"); axis.set_title("Paridade put-call com preços reais")
    axis.legend(); plt.show()
    report("coerência entre calls e puts", [
        f"{len(PARITY)} strikes têm call e put líquidas. **{PARITY['dentro dos limites (mid)'].mean():.0%}** respeitam o intervalo "
        f"americano usando o mid e **{PARITY['dentro com spread'].mean():.0%}** respeitam quando se considera o spread.",
        f"Resíduo europeu mediano: **{PARITY['resíduo europeu'].median():+.3f}** por ação. Resíduos positivos e crescentes "
        "com a strike são esperados em opções americanas (a put americana vale mais que a europeia dentro do dinheiro).",
        "Strikes fora da faixa, com spread considerado, indicam cotações antigas ou dados ruins. Não use essas strikes para "
        "calibrar modelos.",
        "Este teste não depende do Black-Scholes: é uma checagem de qualidade dos dados que todo trabalho com opções deveria fazer.",
    ])

> **Para julgar artigos.** Bons artigos com dados de opções descrevem os **filtros de limpeza**: remoção de opções que violam limites de arbitragem, com preço abaixo de um mínimo (ex.: US$ 0,125), sem negociação ou com spread excessivo. Sem esses filtros, parte dos "resultados" pode ser ruído de dados.

## A9. A put está cara ou barata? Black-Scholes com a volatilidade prevista pelo GARCH

**O que vamos fazer.** Aqui o Black-Scholes deixa de ser "previsor" e vira **régua**:
1. Prevemos a volatilidade até o vencimento com um **GARCH(1,1) com inovações t de Student** (Bollerslev, 1986, 1987), ajustado ao histórico real. O GARCH captura o agrupamento de volatilidade visto em A2:
$$h_{t+1} = \omega + \alpha\,\varepsilon_t^2 + \beta\,h_t$$
A variância média esperada nos próximos $H$ pregões tem forma fechada: com $\bar h = \omega/(1-\alpha-\beta)$,
$$\mathbb{E}[h_{t+k}] = \bar h + (\alpha+\beta)^{k-1}(h_{t+1} - \bar h)$$
2. Calculamos o **preço "justo" do BS** com essa volatilidade prevista.
3. Comparamos com o mid e o ask reais. A diferença em volatilidade é o **prêmio de risco de volatilidade (VRP)**:
$$\text{VRP} = \text{VI}_{\text{mercado}} - \hat\sigma_{\text{GARCH}}$$

Um VRP positivo significa que o mercado cobra mais volatilidade do que o modelo prevê: o seguro está caro em relação ao risco previsto. A literatura documenta que, em média, o VRP é positivo (Carr e Wu, 2009), e esse é um dos motivos de a proteção mecânica com puts render pouco (Israelov, 2019). Na Parte B, esse custo vira um dos sinais da DQN.

In [ ]:
@dataclass
class GarchParameters:
    mean_daily_return: float
    omega: float
    alpha: float
    beta: float
    degrees_of_freedom: float
    next_day_variance: float

    @property
    def persistence(self) -> float:
        return self.alpha + self.beta

    @property
    def long_run_variance(self) -> float:
        return self.omega / (1 - self.persistence)


def fit_garch_t(daily_log_returns: np.ndarray) -> GarchParameters:
    # Máxima verossimilhança do GARCH(1,1) com inovações t de Student (mesmo código do put_de_protecao).
    returns_pct = daily_log_returns * 100

    def unpack(raw):
        mean, log_omega, alpha_logit, beta_logit, log_dof = raw
        alpha = 0.3 / (1 + np.exp(-alpha_logit))
        beta = (0.999 - alpha) / (1 + np.exp(-beta_logit))
        return mean, np.exp(log_omega), alpha, beta, 2.1 + np.exp(log_dof)

    def conditional_variances(mean, omega, alpha, beta):
        shocks = returns_pct - mean
        initial_variance = shocks.var()
        drivers = omega + alpha * np.concatenate([[initial_variance], shocks[:-1] ** 2])
        variances, _ = signal.lfilter([1.0], [1.0, -beta], drivers, zi=[beta * initial_variance])
        return shocks, np.maximum(variances, 1e-8)

    def negative_log_likelihood(raw):
        mean, omega, alpha, beta, dof = unpack(raw)
        shocks, variances = conditional_variances(mean, omega, alpha, beta)
        log_density = (gammaln((dof + 1) / 2) - gammaln(dof / 2) - 0.5 * np.log((dof - 2) * np.pi * variances)
                       - (dof + 1) / 2 * np.log1p(shocks ** 2 / ((dof - 2) * variances)))
        return -log_density.sum()

    start = [returns_pct.mean(), np.log(returns_pct.var() * 0.05), -1.0, 2.0, np.log(4.0)]
    solution = optimize.minimize(negative_log_likelihood, start, method="Nelder-Mead",
                                 options={"maxiter": 4000, "xatol": 1e-6, "fatol": 1e-6})
    mean, omega, alpha, beta, dof = unpack(solution.x)
    shocks, variances = conditional_variances(mean, omega, alpha, beta)
    next_variance = omega + alpha * shocks[-1] ** 2 + beta * variances[-1]
    return GarchParameters(mean / 100, omega / 1e4, alpha, beta, dof, next_variance / 1e4)


def horizon_volatility(garch: GarchParameters, next_day_variance, trading_days: int):
    # Volatilidade anualizada média prevista para os próximos `trading_days` pregões.
    decay = (garch.persistence ** np.arange(trading_days)).mean()
    average_variance = garch.long_run_variance + decay * (np.asarray(next_day_variance) - garch.long_run_variance)
    return np.sqrt(average_variance * TRADING_DAYS_PER_YEAR)


FULL_GARCH = fit_garch_t(LOG_RETURNS.values)
TRADING_DAYS_TO_EXPIRATION = max(int(round(DAYS_TO_EXPIRATION * TRADING_DAYS_PER_YEAR / 365)), 1)
GARCH_VOLATILITY = float(horizon_volatility(FULL_GARCH, FULL_GARCH.next_day_variance, TRADING_DAYS_TO_EXPIRATION))
display(pd.Series({"ω": FULL_GARCH.omega, "α (reação ao choque)": FULL_GARCH.alpha, "β (memória)": FULL_GARCH.beta,
                   "α + β (persistência)": FULL_GARCH.persistence, "graus de liberdade da t": FULL_GARCH.degrees_of_freedom,
                   "vol. de longo prazo (ano)": np.sqrt(FULL_GARCH.long_run_variance * TRADING_DAYS_PER_YEAR),
                   "vol. de amanhã (ano)": np.sqrt(FULL_GARCH.next_day_variance * TRADING_DAYS_PER_YEAR),
                   f"vol. média prevista até o vencimento ({TRADING_DAYS_TO_EXPIRATION} pregões)": GARCH_VOLATILITY},
                  name="GARCH(1,1)-t").to_frame().style.format("{:.4f}"))

In [ ]:
PUTS["preço_bs_garch"] = bs_put_price(SPOT, PUTS["strike"], T_YEARS, RISK_FREE_RATE, DIVIDEND_YIELD, GARCH_VOLATILITY)
PUTS["vrp"] = PUTS["vi_americana_mid"] - GARCH_VOLATILITY
PUTS["ask_vs_justo"] = PUTS["purchase_price"] / PUTS["preço_bs_garch"] - 1
display(PUTS[["strike", "moneyness", "preço_bs_garch", "mid", "purchase_price", "vi_americana_mid", "vrp", "ask_vs_justo"]]
        .rename(columns={"preço_bs_garch": "BS com σ GARCH", "purchase_price": "ask (ou último)",
                         "vi_americana_mid": "VI mercado", "vrp": "VRP (p.p.)", "ask_vs_justo": "ask vs. BS-GARCH"})
        .style.format({"strike": "{:,.2f}", "moneyness": "{:.0%}", "BS com σ GARCH": "{:,.3f}", "mid": "{:,.3f}",
                       "ask (ou último)": "{:,.3f}", "VI mercado": "{:.1%}", "VRP (p.p.)": "{:+.1%}", "ask vs. BS-GARCH": "{:+.0%}"}))

figure, axis = plt.subplots(figsize=(10, 4))
axis.plot(PUTS["moneyness"], PUTS["vi_americana_mid"], "o-", color=COLORS["protected"], label="VI de mercado (mid)")
axis.axhline(GARCH_VOLATILITY, color=COLORS["stock"], ls="--", label=f"σ previsto pelo GARCH ({GARCH_VOLATILITY:.1%})")
axis.axhline(RECENT_VOLATILITY, color=COLORS["accent"], ls=":", label=f"σ histórico 12m ({RECENT_VOLATILITY:.1%})")
axis.fill_between(PUTS["moneyness"], GARCH_VOLATILITY, PUTS["vi_americana_mid"], color=COLORS["protected"], alpha=0.12)
axis.xaxis.set_major_formatter(PercentFormatter(1.0)); axis.yaxis.set_major_formatter(PercentFormatter(1.0))
axis.set_xlabel("strike ÷ preço"); axis.set_title("VRP: a área sombreada é quanto o mercado cobra acima da volatilidade prevista")
axis.legend(); plt.show()

example_index = (PUTS["strike"] - EXAMPLE_PUT.strike).abs().idxmin()
EXAMPLE_VRP = float(PUTS.loc[example_index, "vrp"])
EXAMPLE_MARKUP = float(PUTS.loc[example_index, "ask_vs_justo"])
report("caro ou barato", [
    f"O GARCH tem persistência α + β = **{FULL_GARCH.persistence:.3f}**: choques de volatilidade demoram a se dissipar "
    f"(meia-vida de cerca de {np.log(0.5) / np.log(FULL_GARCH.persistence):.0f} pregões). Graus de liberdade da t = "
    f"{FULL_GARCH.degrees_of_freedom:.1f}: quanto menor, mais gordas as caudas (a normal corresponde a infinito).",
    f"Volatilidade prevista até o vencimento: **{GARCH_VOLATILITY:.1%}**. A put do exemplo é negociada a VI de "
    f"{EXAMPLE_IV:.1%}, então **VRP = {EXAMPLE_VRP * 100:+.1f} p.p.**",
    f"Em preço: o ask da put do exemplo está **{EXAMPLE_MARKUP:+.0%}** em relação ao BS com a volatilidade do GARCH. "
    + ("O seguro está **caro** em relação ao risco que o modelo prevê." if EXAMPLE_MARKUP > 0.05 else
       "O seguro está **perto do justo** ou barato em relação ao risco previsto."),
    f"VRP médio entre as strikes: {PUTS['vrp'].mean() * 100:+.1f} p.p. VRP positivo é o padrão documentado "
    "(Carr e Wu, 2009) e é a explicação econômica de Israelov (2019) para a proteção mecânica render pouco.",
    "**Ligação com a Parte B:** se o custo do seguro varia com o tempo, deve haver dias melhores e piores para comprar. "
    "A DQN recebe esse custo como sinal e tenta aprender quando entrar.",
])

> **Para julgar artigos.** Quando um artigo diz que uma opção está "cara" ou "barata", pergunte: **em relação a quê?** A previsão de volatilidade usada (histórica, GARCH, realizada futura) define a resposta. Usar a volatilidade **realizada futura** (que só se conhece depois) como referência é *look-ahead*: serve para medir o VRP ex post, nunca para uma regra de decisão.

In [ ]:
PART_A_SUMMARY = pd.Series({
    "preço atual": f"{SPOT:,.2f}",
    "put de exemplo": f"strike {EXAMPLE_PUT.strike:,.2f} ({EXAMPLE_PUT.strike / SPOT:.0%}), {DAYS_TO_EXPIRATION} dias",
    "ask pago": f"{EXAMPLE_PREMIUM:,.3f} ({EXAMPLE_PREMIUM / SPOT:.2%} do preço)",
    "perda máxima da posição protegida": f"{max_loss_per_share:,.2f} por ação ({max_loss_per_share / (SPOT + EXAMPLE_PREMIUM):.1%})",
    "BS com vol. histórica 12m": f"{BS_PRICE_HISTORICAL_VOL:,.3f} (σ = {RECENT_VOLATILITY:.1%})",
    "VI de mercado da put": f"{EXAMPLE_IV:.1%}",
    "σ previsto pelo GARCH": f"{GARCH_VOLATILITY:.1%}",
    "VRP da put": f"{EXAMPLE_VRP * 100:+.1f} p.p.",
    "ask vs. BS-GARCH": f"{EXAMPLE_MARKUP:+.0%}",
    "delta da put / da posição": f"{example_delta:+.3f} / {1 + example_delta:.2f}",
    "curtose em excesso dos retornos": f"{excess_kurtosis:.1f}",
}, name="valor")
display(Markdown("### A10. Resumo da Parte A")); display(PART_A_SUMMARY.to_frame())

---
# Parte B — DQN como ponto de entrada da proteção

**A pergunta da Parte B.** Você já decidiu montar a posição protegida (ação + put de strike `protection_moneyness` × preço, prazo `target_protection_days`). Vale montar **hoje** ou esperar alguns dias, até no máximo `entry_window_days` pregões, por um momento melhor?

**Diferença em relação ao Projeto 9 do curso.** No curso, o agente faz **hedge dinâmico**: a cada passo de tempo ele ajusta quantas ações segura para proteger uma opção vendida. Aqui, o agente **compra** a proteção uma única vez e decide **quando**. Os "passos de tempo" são os dias da janela de entrada. A estrutura (dados → MDP → recompensa → memória → rede → política → treino → avaliação) é a mesma do curso.

## B1. Os dados que o agente enxerga

**O que vamos fazer.** Montar, para cada pregão do histórico, um vetor de sinais que usa **só informação disponível naquele dia**:
- **7 sinais da ação** (os mesmos do `put_de_protecao`): retornos de 5 e 20 dias, distância das médias de 20 e 50 dias, RSI de 14 dias, volatilidade de 20 dias ÷ volatilidade de 6 meses e queda desde a máxima de 60 dias.
- **3 sinais do Black-Scholes (Parte A):** custo da proteção (prêmio ÷ preço), volatilidade prevista pelo GARCH e delta da put.

**Sobre o custo histórico da put (leia com atenção).** O Yahoo não guarda preços passados de opções. O prêmio de cada dia do passado é **estimado** pelo BS com:
- a volatilidade prevista pelo GARCH **naquele dia** (ajustado só com dados do período de treino);
- um **fator de calibração** constante = VI real de hoje ÷ volatilidade GARCH de hoje, para que o nível do prêmio estimado corresponda ao que o mercado cobra hoje;
- a taxa `^IRX` real de cada dia.

Consequência: o prêmio estimado acompanha a volatilidade prevista, mas **não** captura variações do VRP ao longo do tempo (o fator é constante). Para um teste com preços reais, troque `estimate_historical_put_premiums` por uma função que leia o prêmio de uma base histórica (COTAHIST, MT5 gravado diariamente, OptionMetrics).

**Separação temporal.** Os primeiros 70% do histórico são de **treino** e os 30% finais de **teste**. Entre os dois há um **embargo** de (janela + prazo da put) pregões, para que nenhum resultado usado no treino dependa de preços do período de teste. O GARCH e a normalização dos sinais usam **só** o período de treino.

In [ ]:
def compute_entry_signals(closes: pd.Series) -> pd.DataFrame:
    # Sinais de mercado usando apenas informação disponível até cada dia.
    log_returns = np.log(closes).diff()
    price_changes = closes.diff()
    average_gain = price_changes.clip(lower=0).ewm(alpha=1 / 14, adjust=False).mean()
    average_loss = (-price_changes.clip(upper=0)).ewm(alpha=1 / 14, adjust=False).mean()
    signals = pd.DataFrame({
        "retorno 5 dias": closes / closes.shift(5) - 1,
        "retorno 20 dias": closes / closes.shift(20) - 1,
        "distância da média 20": closes / closes.rolling(20).mean() - 1,
        "distância da média 50": closes / closes.rolling(50).mean() - 1,
        "RSI 14": 100 - 100 / (1 + average_gain / average_loss.replace(0, np.nan)),
        "vol. 20d / vol. 6 meses": log_returns.rolling(20).std() / log_returns.rolling(126).std(),
        "queda desde a máxima 60d": closes / closes.rolling(60).max() - 1,
    })
    return signals.dropna()


def garch_next_day_variances(garch: GarchParameters, log_returns: pd.Series) -> pd.Series:
    # Variância prevista para o dia seguinte, calculada no fechamento de cada dia (sem olhar o futuro).
    shocks = log_returns.values - garch.mean_daily_return
    drivers = garch.omega + garch.alpha * shocks ** 2
    variances, _ = signal.lfilter([1.0], [1.0, -garch.beta], drivers, zi=[garch.beta * garch.long_run_variance])
    return pd.Series(variances, index=log_returns.index)


def estimate_historical_put_premiums(prices: np.ndarray, rates: np.ndarray, forecast_volatility: np.ndarray,
                                     volatility_markup: float, settings: GuideSettings) -> pd.DataFrame:
    # ESTIMATIVA do prêmio histórico (o Yahoo não tem histórico de opções).
    # Para usar preços reais, substitua esta função por uma leitura de base histórica com as mesmas colunas.
    years = settings.target_protection_days / 365
    strikes = settings.protection_moneyness * prices
    volatility = volatility_markup * forecast_volatility
    return pd.DataFrame({"strike": strikes,
                         "premium": bs_put_price(prices, strikes, years, rates, DIVIDEND_YIELD, volatility),
                         "put_delta": bs_put_delta(prices, strikes, years, rates, DIVIDEND_YIELD, volatility)})


HORIZON_TRADING_DAYS = max(int(round(SETTINGS.target_protection_days * TRADING_DAYS_PER_YEAR / 365)), 1)
WINDOW = SETTINGS.entry_window_days
STOCK_SIGNALS = compute_entry_signals(CLOSES)
DATES = STOCK_SIGNALS.index
PRICES = CLOSES.reindex(DATES).values
TOTAL_DAYS = len(DATES)
TEST_START = int(TOTAL_DAYS * (1 - SETTINGS.test_period_fraction))
TRAINING_END = TEST_START - WINDOW - HORIZON_TRADING_DAYS          # embargo entre treino e teste

TRAINING_GARCH = fit_garch_t(LOG_RETURNS[LOG_RETURNS.index <= DATES[TRAINING_END]].values)
garch_path = horizon_volatility(TRAINING_GARCH, garch_next_day_variances(TRAINING_GARCH, LOG_RETURNS).values, HORIZON_TRADING_DAYS)
GARCH_VOLATILITY_PATH = pd.Series(garch_path, index=LOG_RETURNS.index).reindex(DATES).values
RATES = RATE_HISTORY.reindex(RATE_HISTORY.index.union(DATES)).ffill().bfill().reindex(DATES).values

market_iv_today = interpolate_smile(PUTS, SETTINGS.protection_moneyness)
if not np.isfinite(market_iv_today):
    market_iv_today = interpolate_smile(PUTS, SETTINGS.protection_moneyness, "vi_bs_mid")
VOLATILITY_MARKUP = float(np.clip(market_iv_today / GARCH_VOLATILITY_PATH[-1], 0.5, 3.0)) if np.isfinite(market_iv_today) else 1.0

HISTORICAL_PUTS = estimate_historical_put_premiums(PRICES, RATES, GARCH_VOLATILITY_PATH, VOLATILITY_MARKUP, SETTINGS)
FEATURES = STOCK_SIGNALS.copy()
FEATURES["custo da proteção %"] = 100 * HISTORICAL_PUTS["premium"].values / PRICES
FEATURES["vol. prevista (GARCH)"] = GARCH_VOLATILITY_PATH
FEATURES["delta da put"] = HISTORICAL_PUTS["put_delta"].values
FEATURE_NAMES = list(FEATURES.columns)

ENTRY_COST = PRICES + HISTORICAL_PUTS["premium"].values          # ação + put, por ação
PROTECTED_RETURN = np.full(TOTAL_DAYS, np.nan)                    # resultado da posição protegida no vencimento, em %
valid_days = np.arange(TOTAL_DAYS - HORIZON_TRADING_DAYS)
PROTECTED_RETURN[valid_days] = 100 * (np.maximum(PRICES[valid_days + HORIZON_TRADING_DAYS], HISTORICAL_PUTS["strike"].values[valid_days])
                                      - ENTRY_COST[valid_days]) / ENTRY_COST[valid_days]

FEATURE_MEAN = FEATURES.iloc[:TRAINING_END].mean()
FEATURE_STD = FEATURES.iloc[:TRAINING_END].std().replace(0, 1)
NORMALIZED_FEATURES = ((FEATURES - FEATURE_MEAN) / FEATURE_STD).clip(-5, 5).values

display(FEATURES.describe().T[["mean", "std", "min", "50%", "max"]].style.format("{:,.3f}"))

In [ ]:
figure, (price_axis, cost_axis) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for axis in (price_axis, cost_axis):
    axis.axvspan(DATES[TRAINING_END], DATES[TEST_START], color=COLORS["warning"], alpha=0.25)
    axis.axvspan(DATES[TEST_START], DATES[-1], color=COLORS["grid"], alpha=0.6)
price_axis.plot(DATES, PRICES, color=COLORS["stock"], lw=1.2)
price_axis.set_title(f"{TICKER}: treino (fundo branco), embargo (amarelo) e teste (cinza)"); price_axis.set_ylabel("preço")
cost_axis.plot(DATES, FEATURES["custo da proteção %"], color=COLORS["protected"], lw=1)
cost_axis.set_ylabel("custo da put (% do preço)")
cost_axis.set_title(f"Custo estimado da put {SETTINGS.protection_moneyness:.0%} com {SETTINGS.target_protection_days} dias (BS + GARCH calibrado)")
plt.tight_layout(); plt.show()

correlation = FEATURES.iloc[:TRAINING_END].corr()
figure, axis = plt.subplots(figsize=(8.5, 7))
image = axis.imshow(correlation.values, cmap="RdBu_r", vmin=-1, vmax=1)
axis.set_xticks(range(len(FEATURE_NAMES)), FEATURE_NAMES, rotation=60, ha="right"); axis.set_yticks(range(len(FEATURE_NAMES)), FEATURE_NAMES)
for i in range(len(FEATURE_NAMES)):
    for j in range(len(FEATURE_NAMES)):
        axis.text(j, i, f"{correlation.values[i, j]:.2f}", ha="center", va="center", fontsize=7,
                  color="white" if abs(correlation.values[i, j]) > 0.6 else COLORS["ink_strong"])
axis.grid(False); axis.set_title("Correlação entre os sinais (período de treino)")
figure.colorbar(image, shrink=0.7); plt.show()

high_pairs = [(a, b, correlation.loc[a, b]) for i, a in enumerate(FEATURE_NAMES) for b in FEATURE_NAMES[i + 1:]
              if abs(correlation.loc[a, b]) > 0.8]
report("os dados da Parte B", [
    f"**{TOTAL_DAYS}** pregões com sinais completos: treino até {DATES[TRAINING_END - 1].date()}, embargo de "
    f"{WINDOW + HORIZON_TRADING_DAYS} pregões e teste a partir de {DATES[TEST_START].date()}.",
    f"Fator de calibração da volatilidade: **{VOLATILITY_MARKUP:.2f}** (VI real de hoje {market_iv_today:.1%} ÷ GARCH de hoje "
    f"{GARCH_VOLATILITY_PATH[-1]:.1%}). Todo prêmio histórico é BS × GARCH × esse fator: é uma **estimativa**, não preço real.",
    f"Custo médio estimado da put: {FEATURES['custo da proteção %'].iloc[:TRAINING_END].mean():.2f}% do preço no treino e "
    f"{FEATURES['custo da proteção %'].iloc[TEST_START:].mean():.2f}% no teste.",
    ("Pares de sinais com correlação acima de 0,8: " + "; ".join(f"{a} × {b} ({c:+.2f})" for a, b, c in high_pairs) + ". "
     "Sinais redundantes não atrapalham muito uma rede neural, mas dividem a \"importância\" entre si na seção B9.")
    if high_pairs else "Nenhum par de sinais tem correlação acima de 0,8.",
    "A normalização (média e desvio) usa **só o treino**. Normalizar com o histórico inteiro vazaria informação do futuro.",
])

> **Para julgar artigos.** Três perguntas obrigatórias sobre os dados de um artigo de RL em finanças: (1) os sinais usam **só o passado**? (2) a normalização e os parâmetros (GARCH, médias) foram estimados **só no treino**? (3) há **embargo** entre treino e teste quando a recompensa olha para frente (como aqui, que olha até o vencimento da put)? Se qualquer resposta for "não", os resultados de teste estão inflados.

## B2. O problema como um Processo de Decisão de Markov (MDP)

| Elemento | Definição neste notebook |
|---|---|
| **Episódio** | Uma janela de `entry_window_days` pregões a partir de um dia do histórico. Termina quando o agente compra. |
| **Estado** $s_t$ | Os 10 sinais normalizados do dia + o **tempo restante** na janela (1 no primeiro dia, perto de 0 no último). |
| **Ações** $a_t$ | 0 = **esperar**, 1 = **comprar** ação + put hoje. No último dia, a compra é obrigatória. |
| **Transição** | Se espera, o estado do dia seguinte vem do histórico real (o agente não muda o mercado). |
| **Recompensa** $r_t$ | Zero enquanto espera; ao comprar, o ganho em relação a ter comprado no 1º dia da janela (seção B3). |
| **Desconto** $\gamma$ | 1: o episódio é curto e não faz sentido preferir recompensas próximas. |

O agente aprende a função **Q**: $Q(s, a)$ = recompensa total esperada a partir do estado $s$ ao tomar a ação $a$ e seguir a melhor política depois. A **equação de Bellman** liga um dia ao seguinte:

$$Q(s_t, \text{esperar}) = \mathbb{E}\big[\max_{a'} Q(s_{t+1}, a')\big], \qquad Q(s_t, \text{comprar}) = \mathbb{E}[r_t \mid \text{comprar}]$$

Em palavras: **esperar vale o melhor que se pode fazer amanhã**, e **comprar vale a recompensa de comprar hoje**. A decisão ótima é comprar quando $Q(\text{comprar}) \ge Q(\text{esperar})$. A diferença $A = Q(\text{comprar}) - Q(\text{esperar})$ é a **vantagem** de comprar agora, que usaremos para explicar as decisões.

## B3. As funções de recompensa

**O que vamos fazer.** Implementar duas recompensas (escolha em `reward_type`) e examinar seus números antes de treinar qualquer coisa:

1. **`economia_na_entrada`**: quanto o custo de entrada (ação + put) do dia escolhido ficou abaixo do custo do 1º dia:
$$r = 100 \times \frac{C_{\text{dia 1}} - C_{\text{dia escolhido}}}{C_{\text{dia 1}}}$$
2. **`resultado_protegido`** (padrão): o resultado da posição protegida no vencimento, comprando no dia escolhido, menos o resultado se tivesse comprado no 1º dia:
$$R_d = 100 \times \frac{\max(S_{d+H}, K_d) - C_d}{C_d}, \qquad r = R_{\text{dia escolhido}} - R_{\text{dia 1}}$$
onde $H$ é o prazo da put em pregões. Essa versão é mais fiel ao objetivo (o resultado do hedge), mas tem mais ruído, porque depende de todo o caminho do preço até o vencimento.

Medir a recompensa em relação ao 1º dia faz com que "comprar no 1º dia" valha sempre zero: qualquer média positiva da DQN no teste é ganho sobre a estratégia mais simples.

In [ ]:
ACTION_WAIT, ACTION_BUY = 0, 1


class EntryTimingEnvironment:
    # Episódios de compra sobre o histórico real; o agente não altera o mercado.
    def __init__(self, normalized_features: np.ndarray, entry_cost: np.ndarray, protected_return: np.ndarray,
                 window_days: int, reward_type: str):
        if reward_type not in ("economia_na_entrada", "resultado_protegido"):
            raise ValueError("reward_type deve ser 'economia_na_entrada' ou 'resultado_protegido'")
        self.features, self.entry_cost, self.protected_return = normalized_features, entry_cost, protected_return
        self.window_days, self.reward_type = window_days, reward_type

    @property
    def state_size(self) -> int:
        return self.features.shape[1] + 1

    def states(self, start_days: np.ndarray, days_elapsed: int) -> np.ndarray:
        time_left = np.full((len(start_days), 1), 1 - days_elapsed / self.window_days)
        return np.hstack([self.features[start_days + days_elapsed], time_left])

    def reward(self, start_days: np.ndarray, purchase_days: np.ndarray) -> np.ndarray:
        if self.reward_type == "economia_na_entrada":
            first_day_cost = self.entry_cost[start_days]
            return 100 * (first_day_cost - self.entry_cost[purchase_days]) / first_day_cost
        return self.protected_return[purchase_days] - self.protected_return[start_days]

    def reward_matrix(self, start_days: np.ndarray) -> np.ndarray:
        # Recompensa de comprar em cada dia da janela: linhas = janelas, colunas = dia da compra.
        offsets = np.arange(self.window_days)
        purchase_days = (start_days[:, None] + offsets).ravel()
        return self.reward(np.repeat(start_days, self.window_days), purchase_days).reshape(len(start_days), self.window_days)


ENVIRONMENT = EntryTimingEnvironment(NORMALIZED_FEATURES, ENTRY_COST, PROTECTED_RETURN, WINDOW, SETTINGS.reward_type)
TRAINING_START_DAYS = np.arange(0, TRAINING_END - WINDOW + 1)
TEST_START_DAYS = np.arange(TEST_START, TOTAL_DAYS - WINDOW - HORIZON_TRADING_DAYS + 1, WINDOW)   # janelas sem sobreposição

example_start = int(TRAINING_START_DAYS[len(TRAINING_START_DAYS) // 2])
example_days = example_start + np.arange(WINDOW)
EXAMPLE_WINDOW = pd.DataFrame({
    "data": DATES[example_days].date, "preço": PRICES[example_days],
    "prêmio estimado": HISTORICAL_PUTS["premium"].values[example_days], "custo de entrada": ENTRY_COST[example_days],
    "strike": HISTORICAL_PUTS["strike"].values[example_days],
    "preço no vencimento": PRICES[example_days + HORIZON_TRADING_DAYS],
    "resultado protegido %": PROTECTED_RETURN[example_days],
    "recompensa: economia na entrada": 100 * (ENTRY_COST[example_start] - ENTRY_COST[example_days]) / ENTRY_COST[example_start],
    "recompensa: resultado protegido": PROTECTED_RETURN[example_days] - PROTECTED_RETURN[example_start],
}, index=pd.Index(range(1, WINDOW + 1), name="dia da janela"))
display(Markdown(f"**Uma janela do treino, dia a dia** (início em {DATES[example_start].date()}):"))
display(EXAMPLE_WINDOW.style.format({c: "{:,.2f}" for c in EXAMPLE_WINDOW.columns if c != "data"}))

In [ ]:
training_rewards = ENVIRONMENT.reward_matrix(TRAINING_START_DAYS)
mean_by_day = training_rewards.mean(0)
error_by_day = 1.96 * training_rewards.std(0) / np.sqrt(len(training_rewards))

figure, axis = plt.subplots(figsize=(10, 3.8))
axis.fill_between(np.arange(1, WINDOW + 1), mean_by_day - error_by_day, mean_by_day + error_by_day, color=COLORS["stock"], alpha=0.2)
axis.plot(np.arange(1, WINDOW + 1), mean_by_day, "o-", color=COLORS["stock"])
axis.axhline(0, color=COLORS["ink"], lw=1)
axis.set_xlabel("dia da compra dentro da janela"); axis.set_ylabel("recompensa média (p.p.)")
axis.set_title(f"Recompensa média por dia de compra, todas as janelas do treino ({SETTINGS.reward_type})"); plt.show()

oracle_mean = training_rewards.max(1).mean()
signal_to_noise = np.abs(mean_by_day[1:]).max() / training_rewards[:, 1:].std()
report("o que as recompensas mostram antes de qualquer aprendizado", [
    f"No treino há **{len(TRAINING_START_DAYS)}** janelas possíveis (que se sobrepõem) e no teste **{len(TEST_START_DAYS)}** "
    "janelas sem sobreposição.",
    f"Comprando sempre no mesmo dia, o melhor dia médio é o **{int(mean_by_day.argmax()) + 1}º** "
    f"({mean_by_day.max():+.3f} p.p.). Se esse valor for próximo de zero, esperar não ajuda **em média**; "
    "a DQN só agrega valor se conseguir distinguir **quando** esperar ajuda.",
    f"Um oráculo que soubesse o futuro e escolhesse o melhor dia de cada janela ganharia **{oracle_mean:+.2f} p.p.** em média: "
    "esse é o teto teórico, impossível de alcançar.",
    f"Relação sinal/ruído: o maior efeito médio de esperar é só **{signal_to_noise:.2f}** desvio-padrão da recompensa. "
    "Problemas com sinal tão fraco exigem muitos dados e avaliação estatística cuidadosa: é fácil a rede aprender ruído.",
])

> **Para julgar artigos.** Pergunte (1) **em relação a quê** a recompensa é medida (um benchmark embutido na recompensa, como aqui, deixa a comparação honesta); (2) se a recompensa usa preços **executáveis** (ask para comprar); (3) se o artigo mostra o **teto do oráculo** e a média das estratégias ingênuas. Uma DQN que captura 5% do teto pode ser útil; uma que "ganha" sem comparação com o oráculo e as regras simples não diz nada.

## B4. A memória do agente (experience replay)

**O que vamos fazer.** O agente guarda cada experiência $(s_t, a_t, r_t, s_{t+1}, \text{terminou?})$ numa memória circular e treina com **amostras sorteadas** dela (Lin, 1992; Mnih et al., 2015). Dois motivos:
1. **Quebrar a correlação:** dias seguidos são muito parecidos. Treinar a rede em sequência faz o gradiente "andar" na mesma direção por muitos passos e desestabiliza o aprendizado.
2. **Reaproveitar dados:** cada experiência é usada várias vezes.

A memória também guarda uma marcação **"o próximo dia é o último da janela"**. No último dia só existe a ação comprar, então o valor de esperar até lá é $Q(\text{comprar})$ e **não** $\max_a Q$. Sem essa marcação, a rede supervaloriza esperar (foi o defeito corrigido no `put_de_protecao`).

In [ ]:
class ReplayMemory:
    # Memória circular de transições (s, a, r, s', terminou, próximo é o último dia).
    def __init__(self, capacity: int, state_size: int, seed: int):
        self.capacity, self.rng = capacity, np.random.default_rng(seed)
        self.states = np.zeros((capacity, state_size)); self.next_states = np.zeros((capacity, state_size))
        self.actions = np.zeros(capacity, int); self.rewards = np.zeros(capacity)
        self.done = np.zeros(capacity); self.next_is_last_day = np.zeros(capacity, bool)
        self.write_position, self.size = 0, 0

    def store(self, states, actions, rewards, next_states, done, next_is_last_day) -> None:
        slots = (self.write_position + np.arange(len(states))) % self.capacity
        self.states[slots], self.actions[slots], self.rewards[slots] = states, actions, rewards
        self.next_states[slots], self.done[slots], self.next_is_last_day[slots] = next_states, done, next_is_last_day
        self.write_position = (self.write_position + len(states)) % self.capacity
        self.size = min(self.size + len(states), self.capacity)

    def sample(self, batch_size: int) -> dict:
        index = self.rng.integers(self.size, size=batch_size)
        return {"states": self.states[index], "actions": self.actions[index], "rewards": self.rewards[index],
                "next_states": self.next_states[index], "done": self.done[index], "next_is_last_day": self.next_is_last_day[index]}

    def __len__(self) -> int:
        return self.size


# Demonstração: correlação entre amostras vizinhas, em ordem do tempo × sorteadas
rng = np.random.default_rng(SETTINGS.random_seed)
training_states = NORMALIZED_FEATURES[:TRAINING_END]
shuffled_states = training_states[rng.permutation(len(training_states))]
def neighbour_correlation(matrix, column):
    return np.corrcoef(matrix[:-1, column], matrix[1:, column])[0, 1]
CORRELATION_DEMO = pd.DataFrame({
    "em ordem do tempo": [neighbour_correlation(training_states, j) for j in range(len(FEATURE_NAMES))],
    "sorteadas (replay)": [neighbour_correlation(shuffled_states, j) for j in range(len(FEATURE_NAMES))]},
    index=FEATURE_NAMES)
display(CORRELATION_DEMO.style.format("{:+.2f}"))
report("por que sortear", [
    f"Em ordem do tempo, a correlação média entre um dia e o seguinte é **{CORRELATION_DEMO['em ordem do tempo'].mean():+.2f}**: "
    "um lote de dias seguidos traz quase a mesma informação repetida.",
    f"Depois do sorteio, cai para **{CORRELATION_DEMO['sorteadas (replay)'].mean():+.2f}**: cada lote de treino representa "
    "situações variadas, e o gradiente fica mais próximo do gradiente \"verdadeiro\" do problema.",
])

## B5. A rede Q e a verificação do gradiente

**O que vamos fazer.** A rede recebe o estado (11 números) e devolve dois números, $Q(s, \text{esperar})$ e $Q(s, \text{comprar})$:

$$h_1 = \text{ReLU}(s W_1 + b_1), \quad h_2 = \text{ReLU}(h_1 W_2 + b_2), \quad Q = h_2 W_3 + b_3$$

**Como aprende.** Para cada experiência sorteada, calcula um **alvo** com a equação de Bellman e ajusta os pesos para aproximar $Q(s, a)$ desse alvo:
- Se comprou: alvo = $r$.
- Se esperou: alvo = $Q_{\text{alvo}}(s', a^*)$, com $a^* = \arg\max_a Q_{\text{online}}(s', a)$. Isso é o **Double DQN** (van Hasselt et al., 2016): a rede online **escolhe** a ação e uma cópia atrasada (a **rede-alvo**) **avalia**. Usar a mesma rede para as duas coisas superestima o valor de esperar.

A perda é a **de Huber** (Huber, 1964): quadrática para erros pequenos e linear para erros grandes, o que limita o efeito de recompensas extremas (quedas fortes). O otimizador é o **Adam** (Kingma e Ba, 2015).

A rede está escrita em NumPy, sem biblioteca de deep learning, **de propósito**: cada linha do gradiente fica visível. Para provar que o gradiente está certo, comparamos a retropropagação com **diferenças finitas** em pesos sorteados.

In [ ]:
class QNetwork:
    # MLP com 2 camadas ocultas ReLU; perda de Huber; otimizador Adam.
    def __init__(self, input_size: int, output_size: int, hidden_units: int, learning_rate: float, seed: int):
        rng = np.random.default_rng(seed)
        self.weights = [rng.normal(0, np.sqrt(2 / input_size), (input_size, hidden_units)), np.zeros(hidden_units),
                        rng.normal(0, np.sqrt(2 / hidden_units), (hidden_units, hidden_units)), np.zeros(hidden_units),
                        rng.normal(0, 0.01, (hidden_units, output_size)), np.zeros(output_size)]
        self.first_moments = [np.zeros_like(w) for w in self.weights]
        self.second_moments = [np.zeros_like(w) for w in self.weights]
        self.learning_rate, self.update_count = learning_rate, 0

    def forward(self, states: np.ndarray) -> tuple:
        w1, b1, w2, b2, w3, b3 = self.weights
        hidden1_pre = states @ w1 + b1; hidden1 = np.maximum(hidden1_pre, 0)
        hidden2_pre = hidden1 @ w2 + b2; hidden2 = np.maximum(hidden2_pre, 0)
        return hidden2 @ w3 + b3, (states, hidden1_pre, hidden1, hidden2_pre, hidden2)

    def action_values(self, states: np.ndarray) -> np.ndarray:
        return self.forward(states)[0]

    def loss_and_gradients(self, states, actions, targets) -> tuple:
        predictions, (inputs, hidden1_pre, hidden1, hidden2_pre, hidden2) = self.forward(states)
        batch = np.arange(len(states))
        errors = predictions[batch, actions] - targets
        loss = np.where(np.abs(errors) <= 1, 0.5 * errors ** 2, np.abs(errors) - 0.5).mean()     # Huber
        output_grad = np.zeros_like(predictions)
        output_grad[batch, actions] = np.clip(errors, -1, 1) / len(states)                       # derivada da Huber
        hidden2_grad = (output_grad @ self.weights[4].T) * (hidden2_pre > 0)
        hidden1_grad = (hidden2_grad @ self.weights[2].T) * (hidden1_pre > 0)
        gradients = [inputs.T @ hidden1_grad, hidden1_grad.sum(0), hidden1.T @ hidden2_grad, hidden2_grad.sum(0),
                     hidden2.T @ output_grad, output_grad.sum(0)]
        return float(loss), gradients

    def train_step(self, states, actions, targets) -> float:
        loss, gradients = self.loss_and_gradients(states, actions, targets)
        self.update_count += 1
        for index, gradient in enumerate(gradients):
            self.first_moments[index] = 0.9 * self.first_moments[index] + 0.1 * gradient
            self.second_moments[index] = 0.999 * self.second_moments[index] + 0.001 * gradient ** 2
            corrected_first = self.first_moments[index] / (1 - 0.9 ** self.update_count)
            corrected_second = self.second_moments[index] / (1 - 0.999 ** self.update_count)
            self.weights[index] -= self.learning_rate * corrected_first / (np.sqrt(corrected_second) + 1e-8)
        return loss


def gradient_check(seed: int = 0, checks_per_layer: int = 3, epsilon: float = 1e-6) -> pd.DataFrame:
    # Compara a retropropagação com diferenças finitas numa rede pequena com dados sorteados.
    rng = np.random.default_rng(seed)
    network = QNetwork(ENVIRONMENT.state_size, 2, 8, 1e-3, seed)
    network.weights[4] = rng.normal(0, 0.5, network.weights[4].shape)          # pesos de saída maiores: gradiente visível
    network.weights[1] = rng.normal(0, 0.1, network.weights[1].shape)          # vieses ≠ 0: evita ReLU exatamente em 0
    network.weights[3] = rng.normal(0, 0.1, network.weights[3].shape)
    states = rng.normal(size=(32, ENVIRONMENT.state_size)); actions = rng.integers(2, size=32); targets = rng.normal(size=32)
    _, analytic = network.loss_and_gradients(states, actions, targets)
    layer_names = ["W1", "b1", "W2", "b2", "W3", "b3"]
    rows = []
    for layer, name in enumerate(layer_names):
        for _ in range(checks_per_layer):
            position = tuple(int(rng.integers(dim)) for dim in network.weights[layer].shape)
            original = network.weights[layer][position]
            network.weights[layer][position] = original + epsilon
            loss_up, _ = network.loss_and_gradients(states, actions, targets)
            network.weights[layer][position] = original - epsilon
            loss_down, _ = network.loss_and_gradients(states, actions, targets)
            network.weights[layer][position] = original
            numeric = (loss_up - loss_down) / (2 * epsilon)
            exact = analytic[layer][position]
            rows.append({"camada": name, "posição": str(position), "retropropagação": exact, "diferenças finitas": numeric,
                         "erro relativo": abs(exact - numeric) / max(abs(exact) + abs(numeric), 1e-12)})
    return pd.DataFrame(rows)


GRADIENT_CHECK = gradient_check()
display(GRADIENT_CHECK.style.format({"retropropagação": "{:+.6e}", "diferenças finitas": "{:+.6e}", "erro relativo": "{:.1e}"}))
report("o gradiente está certo?", [
    f"Maior erro relativo entre retropropagação e diferenças finitas: **{GRADIENT_CHECK['erro relativo'].max():.1e}**. "
    "Valores abaixo de 1e-5 indicam gradiente correto (diferenças maiores costumam aparecer só quando um neurônio ReLU "
    "está exatamente na fronteira entre ligado e desligado).",
    f"A rede tem {sum(w.size for w in QNetwork(ENVIRONMENT.state_size, 2, SETTINGS.dqn_hidden_units, 1e-3, 0).weights):,} "
    f"parâmetros para aprender com {len(TRAINING_START_DAYS):,} janelas de treino que se sobrepõem.",
])

> **Para julgar artigos.** Verifique se o artigo informa: arquitetura (camadas, neurônios), perda (MSE ou Huber), otimizador e taxa de aprendizado, frequência de atualização da rede-alvo, tamanho da memória e do lote, e se usa Double DQN. Sem isso o resultado não é reproduzível. Desconfie de redes com muito mais parâmetros do que exemplos independentes de treino.

## B6. A política: explorar ou aproveitar

**O que vamos fazer.** Durante o treino, o agente segue uma política **ε-greedy**: com probabilidade $\varepsilon$ escolhe uma ação ao acaso (**explora**); caso contrário, escolhe a ação de maior Q (**aproveita**). O $\varepsilon$ começa em 1 (tudo ao acaso, porque a rede ainda não sabe nada) e cai linearmente até 0,05 em 70% das rodadas. Na avaliação, a política é **gulosa** ($\varepsilon = 0$): compra no primeiro dia em que $Q(\text{comprar}) \ge Q(\text{esperar})$.

In [ ]:
@dataclass(frozen=True)
class EpsilonGreedyPolicy:
    initial_rate: float = 1.0
    final_rate: float = 0.05
    decay_fraction: float = 0.7          # fração das rodadas em que ε cai até o valor final

    def exploration_rate(self, training_round: int, total_rounds: int) -> float:
        progress = training_round / (self.decay_fraction * total_rounds)
        return max(self.final_rate, self.initial_rate - (self.initial_rate - self.final_rate) * progress)

    def choose_actions(self, q_values: np.ndarray, exploration_rate: float, rng: np.random.Generator,
                       force_buy: bool = False) -> np.ndarray:
        greedy = q_values.argmax(1)
        explore = rng.random(len(q_values)) < exploration_rate
        actions = np.where(explore, rng.integers(2, size=len(q_values)), greedy)
        if force_buy:
            actions[:] = ACTION_BUY
        return actions


POLICY = EpsilonGreedyPolicy()
rounds = np.arange(SETTINGS.dqn_training_rounds)
figure, axis = plt.subplots(figsize=(10, 2.8))
axis.plot(rounds, [POLICY.exploration_rate(r, SETTINGS.dqn_training_rounds) for r in rounds], color=COLORS["purple"])
axis.yaxis.set_major_formatter(PercentFormatter(1.0)); axis.set_xlabel("rodada de treino"); axis.set_ylabel("ε")
axis.set_title("Taxa de exploração ao longo do treino"); plt.show()

## B7. Treinamento com várias sementes

**O que vamos fazer.** Treinar a DQN uma vez para cada semente aleatória em `dqn_seeds`. Cada rodada:
1. sorteia 64 inícios de janela do período de treino;
2. percorre os dias da janela: o agente escolhe esperar ou comprar pela política ε-greedy; no último dia compra obrigatoriamente;
3. guarda cada transição na memória;
4. faz 4 atualizações da rede com lotes de 256 transições sorteadas;
5. a cada 200 atualizações, copia a rede online para a rede-alvo.

**Por que várias sementes.** Em RL, a mesma configuração pode dar resultados bem diferentes só por causa da sorte na inicialização e nos sorteios (Henderson et al., 2018). Um resultado só é confiável se aparecer na **maioria** das sementes.

In [ ]:
def train_entry_dqn(environment: EntryTimingEnvironment, training_start_days: np.ndarray, settings: GuideSettings,
                    seed: int, policy: EpsilonGreedyPolicy = POLICY) -> tuple:
    rng = np.random.default_rng(seed)
    online_network = QNetwork(environment.state_size, 2, settings.dqn_hidden_units, 1e-3, seed)
    target_network = copy.deepcopy(online_network)
    memory = ReplayMemory(100_000, environment.state_size, seed)
    parallel_episodes, batch_size, target_sync_every = 64, 256, 200
    history = []

    for training_round in range(settings.dqn_training_rounds):
        exploration_rate = policy.exploration_rate(training_round, settings.dqn_training_rounds)
        start_days = rng.choice(training_start_days, parallel_episodes)
        still_waiting = np.ones(parallel_episodes, bool)
        episode_rewards = np.zeros(parallel_episodes)
        purchase_offsets = np.full(parallel_episodes, environment.window_days - 1)
        round_losses = []
        for days_elapsed in range(environment.window_days):
            active = np.where(still_waiting)[0]
            if not len(active):
                break
            states = environment.states(start_days[active], days_elapsed)
            last_day = days_elapsed == environment.window_days - 1
            actions = policy.choose_actions(online_network.action_values(states), exploration_rate, rng, force_buy=last_day)
            bought = actions == ACTION_BUY
            rewards = np.where(bought, environment.reward(start_days[active], start_days[active] + days_elapsed), 0.0)
            next_states = states if last_day else environment.states(start_days[active], days_elapsed + 1)
            memory.store(states, actions, rewards, next_states, bought.astype(float),
                         days_elapsed + 1 == environment.window_days - 1)
            episode_rewards[active] += rewards
            purchase_offsets[active[bought]] = days_elapsed
            still_waiting[active[bought]] = False

            if len(memory) >= 1_000:
                for _ in range(4):
                    batch = memory.sample(batch_size)
                    # Double DQN: a rede online escolhe a próxima ação; a rede-alvo avalia.
                    best_next_actions = online_network.action_values(batch["next_states"]).argmax(1)
                    best_next_actions[batch["next_is_last_day"]] = ACTION_BUY        # no último dia só existe comprar
                    next_values = target_network.action_values(batch["next_states"])[np.arange(batch_size), best_next_actions]
                    targets = batch["rewards"] + (1 - batch["done"]) * next_values
                    round_losses.append(online_network.train_step(batch["states"], batch["actions"], targets))
                    if online_network.update_count % target_sync_every == 0:
                        target_network = copy.deepcopy(online_network)
        history.append({"rodada": training_round, "ε": exploration_rate, "recompensa média": episode_rewards.mean(),
                        "dia médio de compra": purchase_offsets.mean() + 1,
                        "perda média": np.mean(round_losses) if round_losses else np.nan})
    return online_network, pd.DataFrame(history)


TRAINED_AGENTS = {}
for seed in SETTINGS.dqn_seeds:
    network, history = train_entry_dqn(ENVIRONMENT, TRAINING_START_DAYS, SETTINGS, seed)
    TRAINED_AGENTS[seed] = {"network": network, "history": history}
print(f"Treinadas {len(TRAINED_AGENTS)} redes ({SETTINGS.dqn_training_rounds} rodadas × 64 episódios cada).")

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for seed, agent in TRAINED_AGENTS.items():
    history = agent["history"]
    axes[0].plot(history["rodada"], history["recompensa média"].rolling(10, min_periods=1).mean(), lw=1.2, label=f"semente {seed}")
    axes[1].plot(history["rodada"], history["perda média"].rolling(10, min_periods=1).mean(), lw=1.2)
    axes[2].plot(history["rodada"], history["dia médio de compra"].rolling(10, min_periods=1).mean(), lw=1.2)
axes[0].axhline(0, color=COLORS["ink"], lw=1)
axes[0].set_title("Recompensa média por episódio (média móvel 10)"); axes[0].legend(fontsize=7)
axes[1].set_title("Perda de Huber"); axes[2].set_title("Dia médio de compra na janela")
for axis in axes:
    axis.set_xlabel("rodada")
plt.tight_layout(); plt.show()

final_phase = int(SETTINGS.dqn_training_rounds * 0.8)
final_rewards = {seed: agent["history"]["recompensa média"].iloc[final_phase:].mean() for seed, agent in TRAINED_AGENTS.items()}
final_days = {seed: agent["history"]["dia médio de compra"].iloc[final_phase:].mean() for seed, agent in TRAINED_AGENTS.items()}
report("o agente aprendeu alguma coisa no treino?", [
    "No início (ε alto) o agente compra em dias aleatórios. Se ele aprende, a recompensa média sobe conforme o ε cai.",
    "Recompensa média nos últimos 20% das rodadas (ainda com 5% de exploração), por semente: "
    + ", ".join(f"semente {s}: **{v:+.3f}**" for s, v in final_rewards.items()) + " p.p.",
    "Dia médio de compra no fim do treino: " + ", ".join(f"{s}: {v:.1f}" for s, v in final_days.items()) +
    ". Se uma semente compra sempre no dia 1 ou sempre no último dia, a política dela é **degenerada**: aprendeu uma "
    "regra fixa, não um momento de entrada.",
    "**Atenção:** recompensa alta no treino não prova nada. As janelas de treino se sobrepõem e a rede pode decorar "
    "o histórico. O teste que importa é o da seção B8, num período que a rede nunca viu.",
])

> **Para julgar artigos.** Exija: (1) curvas de aprendizado; (2) **número de sementes** e a dispersão entre elas (média e intervalo, não a melhor execução); (3) a **política aprendida** (em que dia compra, com que frequência faz cada ação). Uma política que sempre faz a mesma coisa é uma regra fixa disfarçada.

## B8. Avaliação fora da amostra, passo a passo

**O que vamos fazer.** Rodar cada rede treinada, sem exploração, nas janelas **sem sobreposição** do período de teste, e comparar com cinco referências:

| Estratégia | Regra |
|---|---|
| Comprar no 1º dia | não espera nunca (recompensa = 0 por construção) |
| Preço médio da janela | compra 1/N da posição por dia |
| Dia aleatório | sorteia o dia |
| **Regra simples explicável** | compra no 1º dia em que o custo da proteção estiver abaixo da mediana do treino; senão, no último dia |
| Oráculo | o melhor dia de cada janela, sabendo o futuro (teto impossível) |

A regra simples é a referência mais importante: se a DQN não superar uma regra de uma linha que usa o mesmo sinal, a complexidade não se justifica.

**Estatística.** Para cada semente: média da recompensa, intervalo de confiança por **bootstrap** (Efron, 1979), teste t contra zero (o 1º dia) e teste t pareado contra a regra simples e o preço médio.

In [ ]:
def greedy_purchase_days(network: QNetwork, environment: EntryTimingEnvironment, start_days: np.ndarray) -> np.ndarray:
    # Compra no primeiro dia em que Q(comprar) ≥ Q(esperar); se nunca, no último dia.
    purchase_days = start_days + environment.window_days - 1
    decided = np.zeros(len(start_days), bool)
    for days_elapsed in range(environment.window_days - 1):
        undecided = np.where(~decided)[0]
        values = network.action_values(environment.states(start_days[undecided], days_elapsed))
        buys_now = undecided[values[:, ACTION_BUY] >= values[:, ACTION_WAIT]]
        purchase_days[buys_now] = start_days[buys_now] + days_elapsed
        decided[buys_now] = True
    return purchase_days


def bootstrap_mean_interval(values: np.ndarray, rng: np.random.Generator, draws: int = 5_000) -> tuple:
    means = rng.choice(values, (draws, len(values))).mean(1)
    return tuple(np.percentile(means, [2.5, 97.5]))


RULE_THRESHOLD = float(FEATURES["custo da proteção %"].iloc[:TRAINING_END].median())
RAW_PROTECTION_COST = FEATURES["custo da proteção %"].values
TEST_REWARDS = ENVIRONMENT.reward_matrix(TEST_START_DAYS)
window_rows = np.arange(len(TEST_START_DAYS))
evaluation_rng = np.random.default_rng(SETTINGS.random_seed + 1)
cheap_days = RAW_PROTECTION_COST[TEST_START_DAYS[:, None] + np.arange(WINDOW)] <= RULE_THRESHOLD
rule_offsets = np.where(cheap_days.any(1), cheap_days.argmax(1), WINDOW - 1)
BENCHMARKS = {
    "Comprar no 1º dia": TEST_REWARDS[:, 0],
    "Preço médio da janela": TEST_REWARDS.mean(1),
    "Dia aleatório": TEST_REWARDS[window_rows, evaluation_rng.integers(WINDOW, size=len(TEST_START_DAYS))],
    "Regra simples (custo < mediana)": TEST_REWARDS[window_rows, rule_offsets],
    "Oráculo (teto)": TEST_REWARDS.max(1),
}

seed_rows, DQN_REWARDS, DQN_OFFSETS = [], {}, {}
for seed, agent in TRAINED_AGENTS.items():
    offsets = greedy_purchase_days(agent["network"], ENVIRONMENT, TEST_START_DAYS) - TEST_START_DAYS
    rewards = TEST_REWARDS[window_rows, offsets]
    DQN_REWARDS[seed], DQN_OFFSETS[seed] = rewards, offsets
    lower, upper = bootstrap_mean_interval(rewards, evaluation_rng)
    seed_rows.append({"semente": seed, "recompensa média": rewards.mean(), "IC 95% inferior": lower, "IC 95% superior": upper,
                      "p vs. 1º dia": stats.ttest_1samp(rewards, 0).pvalue if rewards.std() > 0 else 1.0,
                      "p vs. regra simples": stats.ttest_rel(rewards, BENCHMARKS["Regra simples (custo < mediana)"]).pvalue,
                      "p vs. preço médio": stats.ttest_rel(rewards, BENCHMARKS["Preço médio da janela"]).pvalue,
                      "dia médio de compra": offsets.mean() + 1, "% compras no 1º dia": (offsets == 0).mean()})
EVALUATION_BY_SEED = pd.DataFrame(seed_rows).set_index("semente")
display(Markdown(f"**DQN por semente**: {len(TEST_START_DAYS)} janelas de teste desde {DATES[TEST_START].date()}; recompensa = `{SETTINGS.reward_type}` (p.p.)"))
display(EVALUATION_BY_SEED.style.format({"recompensa média": "{:+.3f}", "IC 95% inferior": "{:+.3f}", "IC 95% superior": "{:+.3f}",
                                         "p vs. 1º dia": "{:.3f}", "p vs. regra simples": "{:.3f}", "p vs. preço médio": "{:.3f}",
                                         "dia médio de compra": "{:.1f}", "% compras no 1º dia": "{:.0%}"}))

MEDIAN_SEED = EVALUATION_BY_SEED["recompensa média"].sort_values().index[len(EVALUATION_BY_SEED) // 2]
summary_rows = {}
for name, rewards in {f"DQN (semente mediana: {MEDIAN_SEED})": DQN_REWARDS[MEDIAN_SEED], **BENCHMARKS}.items():
    lower, upper = bootstrap_mean_interval(rewards, evaluation_rng)
    summary_rows[name] = {"recompensa média": rewards.mean(), "IC 95% inferior": lower, "IC 95% superior": upper,
                          "% janelas acima do 1º dia": (rewards > 0).mean()}
STRATEGY_SUMMARY = pd.DataFrame(summary_rows).T
display(STRATEGY_SUMMARY.style.format({"recompensa média": "{:+.3f}", "IC 95% inferior": "{:+.3f}",
                                       "IC 95% superior": "{:+.3f}", "% janelas acima do 1º dia": "{:.0%}"}))

In [ ]:
figure, axis = plt.subplots(figsize=(10, 4))
plot_table = STRATEGY_SUMMARY.iloc[::-1]
for position, (name, row) in enumerate(plot_table.iterrows()):
    color = COLORS["protected"] if name.startswith("DQN") else COLORS["muted"] if name.startswith("Oráculo") else COLORS["stock"]
    axis.plot([row["IC 95% inferior"], row["IC 95% superior"]], [position, position], color=color, lw=2)
    axis.plot(row["recompensa média"], position, "o", color=color, ms=8, mec=COLORS["surface"], mew=2)
dqn_position = len(plot_table) - 1
for seed, rewards in DQN_REWARDS.items():
    axis.plot(rewards.mean(), dqn_position, "|", color=COLORS["protected"], ms=14, alpha=0.6)
axis.axvline(0, color=COLORS["ink"], lw=1)
axis.set_yticks(range(len(plot_table)), plot_table.index); axis.grid(axis="y", visible=False)
axis.set_xlabel("recompensa média no teste (p.p.), com IC de 95%; tracinhos = cada semente da DQN")
axis.set_title("Fora da amostra: DQN contra as referências"); plt.show()

rule_mean = BENCHMARKS["Regra simples (custo < mediana)"].mean()
average_mean = BENCHMARKS["Preço médio da janela"].mean()
seed_beats = ((EVALUATION_BY_SEED["recompensa média"] > 0) & (EVALUATION_BY_SEED["p vs. 1º dia"] < 0.05)
              & (EVALUATION_BY_SEED["recompensa média"] > rule_mean) & (EVALUATION_BY_SEED["recompensa média"] > average_mean))
seeds_beating = int(seed_beats.sum())
if seeds_beating >= int(np.ceil(0.8 * len(EVALUATION_BY_SEED))):
    DQN_VERDICT = "consistente: supera as referências com significância em quase todas as sementes"
elif seeds_beating > 0:
    DQN_VERDICT = "instável: supera as referências só em algumas sementes (depende da sorte)"
else:
    DQN_VERDICT = "não superou as referências com significância"
oracle_capture = DQN_REWARDS[MEDIAN_SEED].mean() / BENCHMARKS["Oráculo (teto)"].mean() if BENCHMARKS["Oráculo (teto)"].mean() > 0 else np.nan
report("a DQN funciona como ponto de entrada?", [
    f"Veredito: **{DQN_VERDICT}** ({seeds_beating} de {len(EVALUATION_BY_SEED)} sementes passam em todos os critérios: média "
    "positiva, p < 0,05 contra o 1º dia e média acima da regra simples e do preço médio).",
    f"DQN (semente mediana): **{DQN_REWARDS[MEDIAN_SEED].mean():+.3f} p.p.** por janela. Regra simples: {rule_mean:+.3f}. "
    f"Preço médio: {average_mean:+.3f}. Oráculo: {BENCHMARKS['Oráculo (teto)'].mean():+.3f}.",
    f"A DQN captura **{oracle_capture:.0%}** do ganho máximo teórico (oráculo)." if np.isfinite(oracle_capture) else
    "O oráculo não tem ganho positivo no teste.",
    f"Espalhamento entre sementes: de {EVALUATION_BY_SEED['recompensa média'].min():+.3f} a "
    f"{EVALUATION_BY_SEED['recompensa média'].max():+.3f} p.p. Se esse intervalo cruza o zero, o sinal da DQN é fraco.",
    "Lembre: o prêmio da put no histórico é **estimado** (BS + GARCH calibrado). Este resultado mede se a DQN aprende a "
    "explorar a volatilidade prevista e os sinais da ação; não mede se ela capturaria variações reais do VRP.",
])

In [ ]:
median_network = TRAINED_AGENTS[MEDIAN_SEED]["network"]
median_offsets = DQN_OFFSETS[MEDIAN_SEED]
illustration = int(np.argmax(median_offsets > 0)) if (median_offsets > 0).any() else 0
start = TEST_START_DAYS[illustration]
window_days = start + np.arange(WINDOW)
q_values = np.vstack([median_network.action_values(ENVIRONMENT.states(np.array([start]), k)) for k in range(WINDOW)])
purchase_offset = int(median_offsets[illustration])
STEP_TABLE = pd.DataFrame({
    "data": DATES[window_days].date, "preço": PRICES[window_days], "custo da proteção %": RAW_PROTECTION_COST[window_days],
    "vol. prevista": GARCH_VOLATILITY_PATH[window_days], "Q(esperar)": q_values[:, ACTION_WAIT], "Q(comprar)": q_values[:, ACTION_BUY],
    "vantagem de comprar": q_values[:, ACTION_BUY] - q_values[:, ACTION_WAIT],
    "decisão": ["esperar" if k < purchase_offset else "**COMPRAR**" if k == purchase_offset else "(já comprou)" for k in range(WINDOW)],
    "recompensa se comprasse": TEST_REWARDS[illustration],
}, index=pd.Index(range(1, WINDOW + 1), name="dia"))
display(Markdown(f"**Um episódio de teste, passo a passo** (semente {MEDIAN_SEED}, janela que começa em {DATES[start].date()}):"))
display(STEP_TABLE.style.format({"preço": "{:,.2f}", "custo da proteção %": "{:.2f}", "vol. prevista": "{:.1%}",
                                 "Q(esperar)": "{:+.3f}", "Q(comprar)": "{:+.3f}", "vantagem de comprar": "{:+.3f}",
                                 "recompensa se comprasse": "{:+.2f}"}))

figure, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].plot(range(1, WINDOW + 1), q_values[:, ACTION_WAIT], "o-", color=COLORS["stock"], label="Q(esperar)")
axes[0].plot(range(1, WINDOW + 1), q_values[:, ACTION_BUY], "o-", color=COLORS["protected"], label="Q(comprar)")
axes[0].axvline(purchase_offset + 1, color=COLORS["ink"], ls=":", lw=1)
axes[0].set_title("Q-valores no episódio ilustrado"); axes[0].set_xlabel("dia da janela"); axes[0].legend()
test_states_by_day = [ENVIRONMENT.states(TEST_START_DAYS, k) for k in range(WINDOW)]
mean_advantage = [np.mean(np.diff(median_network.action_values(states), axis=1)) for states in test_states_by_day]
axes[1].bar(range(1, WINDOW + 1), mean_advantage, color=COLORS["purple"])
axes[1].axhline(0, color=COLORS["ink"], lw=1)
axes[1].set_title("Vantagem média de comprar, por dia (teste)"); axes[1].set_xlabel("dia da janela")
width = 0.8 / len(DQN_OFFSETS)
for index, (seed, offsets) in enumerate(DQN_OFFSETS.items()):
    counts = np.bincount(offsets, minlength=WINDOW) / len(offsets)
    axes[2].bar(np.arange(1, WINDOW + 1) + (index - len(DQN_OFFSETS) / 2) * width, counts, width=width, label=f"semente {seed}")
axes[2].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[2].set_title("Em que dia cada semente compra (teste)"); axes[2].set_xlabel("dia da janela"); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

report("lendo um episódio", [
    f"No episódio ilustrado, a rede comprou no **dia {purchase_offset + 1}**, o primeiro em que Q(comprar) ≥ Q(esperar). "
    f"A recompensa obtida foi **{TEST_REWARDS[illustration, purchase_offset]:+.2f} p.p.**; o melhor dia possível daria "
    f"{TEST_REWARDS[illustration].max():+.2f} p.p.",
    "Os Q-valores estão na mesma unidade da recompensa (p.p.). Q(esperar) é a previsão da rede do que se ganha esperando e "
    "agindo bem depois; Q(comprar) é a previsão do ganho de comprar hoje.",
    "No gráfico do meio, vantagem negativa nos primeiros dias significa que a rede tende a esperar; perto do fim da janela, "
    "a vantagem precisa subir, porque a opção de esperar vai se esgotando.",
    "No gráfico da direita, sementes que compram quase sempre no mesmo dia aprenderam uma regra fixa; sementes que "
    "espalham as compras estão reagindo ao estado do mercado.",
])

> **Para julgar artigos.** Desconfie de artigos de RL em finanças que (1) comparam só com *buy-and-hold*; (2) não mostram **intervalo de confiança** nem número de janelas independentes de teste; (3) reportam só a melhor semente; (4) não incluem uma **regra simples** com os mesmos sinais. Um resultado robusto aparece em várias sementes e supera a regra simples com significância.

## B9. Explicabilidade: por que a rede compra ou espera?

**O que vamos fazer.** Abrir a rede com duas técnicas independentes de modelo:
1. **Importância por permutação** (Breiman, 2001): embaralhar um sinal entre os estados de teste, quebrando a relação dele com o resto, e medir quantas decisões mudam. Quanto mais decisões mudam, mais a rede depende daquele sinal.
2. **Dependência parcial** (Friedman, 2001): variar um sinal numa grade de valores, mantendo os outros como estão, e medir a vantagem média de comprar, $A = Q(\text{comprar}) - Q(\text{esperar})$. A curva mostra **em que direção** o sinal empurra a decisão.

Usamos a rede da semente mediana e todos os estados do período de teste (cada dia de cada janela).

In [ ]:
STATE_NAMES = FEATURE_NAMES + ["tempo restante na janela"]
EXPLAIN_STATES = np.vstack(test_states_by_day)


def buy_advantage(network: QNetwork, states: np.ndarray) -> np.ndarray:
    values = network.action_values(states)
    return values[:, ACTION_BUY] - values[:, ACTION_WAIT]


def permutation_importance(network: QNetwork, states: np.ndarray, rng: np.random.Generator, repeats: int = 5) -> pd.DataFrame:
    base_advantage = buy_advantage(network, states)
    base_decision = base_advantage >= 0
    rows = []
    for column, name in enumerate(STATE_NAMES):
        flips, shifts = [], []
        for _ in range(repeats):
            shuffled = states.copy()
            shuffled[:, column] = rng.permutation(shuffled[:, column])
            advantage = buy_advantage(network, shuffled)
            flips.append(((advantage >= 0) != base_decision).mean())
            shifts.append(np.abs(advantage - base_advantage).mean())
        rows.append({"sinal": name, "% decisões que mudam": np.mean(flips), "mudança média da vantagem (p.p.)": np.mean(shifts)})
    return pd.DataFrame(rows).set_index("sinal").sort_values("% decisões que mudam", ascending=False)


def partial_dependence(network: QNetwork, states: np.ndarray, column: int, points: int = 20) -> tuple:
    grid = np.quantile(states[:, column], np.linspace(0.05, 0.95, points))
    averages = []
    for value in grid:
        modified = states.copy(); modified[:, column] = value
        averages.append(buy_advantage(network, modified).mean())
    return grid, np.array(averages)


IMPORTANCE = permutation_importance(median_network, EXPLAIN_STATES, np.random.default_rng(SETTINGS.random_seed))
display(IMPORTANCE.style.format({"% decisões que mudam": "{:.1%}", "mudança média da vantagem (p.p.)": "{:.3f}"}))

figure, axis = plt.subplots(figsize=(10, 4))
ordered = IMPORTANCE.iloc[::-1]
axis.barh(ordered.index, ordered["% decisões que mudam"], color=COLORS["stock"])
axis.xaxis.set_major_formatter(PercentFormatter(1.0))
axis.set_title(f"Importância por permutação (semente {MEDIAN_SEED}): % de decisões que mudam ao embaralhar o sinal")
axis.grid(axis="y", visible=False); plt.show()

top_signals = list(IMPORTANCE.index[:3])
figure, axes = plt.subplots(1, 3, figsize=(15, 3.6))
directions = {}
for axis, name in zip(axes, top_signals):
    column = STATE_NAMES.index(name)
    grid, averages = partial_dependence(median_network, EXPLAIN_STATES, column)
    original_units = grid if name == "tempo restante na janela" else grid * FEATURE_STD[name] + FEATURE_MEAN[name]
    axis.plot(original_units, averages, "o-", color=COLORS["protected"], ms=4)
    axis.axhline(0, color=COLORS["ink"], lw=1)
    axis.set_title(f"Dependência parcial: {name}"); axis.set_xlabel(name); axis.set_ylabel("vantagem média de comprar")
    directions[name] = averages[-1] - averages[0]
plt.tight_layout(); plt.show()

def describe_direction(name: str, change: float) -> str:
    tendency = "mais a rede quer **comprar já**" if change > 0 else "mais a rede prefere **esperar**"
    return f"**{name}**: quanto maior, {tendency} (a vantagem muda {change:+.3f} p.p. do menor para o maior valor)."

report("o que move as decisões", [
    f"O sinal mais importante é **{IMPORTANCE.index[0]}**: embaralhá-lo muda {IMPORTANCE.iloc[0, 0]:.0%} das decisões. "
    f"Em seguida vêm {IMPORTANCE.index[1]} ({IMPORTANCE.iloc[1, 0]:.0%}) e {IMPORTANCE.index[2]} ({IMPORTANCE.iloc[2, 0]:.0%}).",
    *[describe_direction(name, change) for name, change in directions.items()],
    "Confira se essas direções fazem **sentido econômico**. Por exemplo: comprar quando o custo da proteção está baixo, ou "
    "esperar depois de uma alta forte. Uma direção sem explicação plausível é sinal de que a rede aprendeu ruído do treino.",
    "Sinais muito correlacionados (seção B1) dividem a importância: embaralhar um deles pouco muda se o outro carrega a "
    "mesma informação. Por isso a importância baixa de um sinal não prova que ele é inútil.",
])

> **Para julgar artigos.** Um artigo que propõe um agente para decisões financeiras deveria mostrar **o que o agente aprendeu**: importância dos sinais, direção dos efeitos, exemplos de decisões. Sem isso, não há como saber se o resultado vem de uma relação econômica estável ou de uma coincidência do período de teste.

## B10. Relatório final

In [ ]:
FINAL_REPORT = pd.Series({
    "ativo": TICKER,
    "put estudada": f"strike {EXAMPLE_PUT.strike:,.2f} ({EXAMPLE_PUT.strike / SPOT:.0%}), {DAYS_TO_EXPIRATION} dias, ask {EXAMPLE_PREMIUM:,.3f}",
    "perda máxima da posição protegida": f"{max_loss_per_share / (SPOT + EXAMPLE_PREMIUM):.1%} do valor investido",
    "VI de mercado / σ GARCH / VRP": f"{EXAMPLE_IV:.1%} / {GARCH_VOLATILITY:.1%} / {EXAMPLE_VRP * 100:+.1f} p.p.",
    "ask vs. BS-GARCH (caro ou barato)": f"{EXAMPLE_MARKUP:+.0%}",
    "recompensa da DQN": SETTINGS.reward_type,
    "período de teste": f"{DATES[TEST_START].date()} a {DATES[-1].date()} ({len(TEST_START_DAYS)} janelas independentes)",
    "DQN (semente mediana)": f"{DQN_REWARDS[MEDIAN_SEED].mean():+.3f} p.p. por janela",
    "regra simples / preço médio / oráculo": f"{rule_mean:+.3f} / {average_mean:+.3f} / {BENCHMARKS['Oráculo (teto)'].mean():+.3f} p.p.",
    "veredito da DQN": DQN_VERDICT,
    "sinal mais importante": IMPORTANCE.index[0],
    "origem do prêmio histórico": "ESTIMADO (BS + GARCH calibrado à VI de hoje)",
}, name="resultado")
display(FINAL_REPORT.to_frame())

### Checklist para julgar artigos (e o seu próprio trabalho)

**Black-Scholes e opções**
- [ ] Qual preço da opção foi usado: ask, mid, fechamento ou teórico? Com que horário?
- [ ] Opções europeias ou americanas? O modelo é compatível?
- [ ] De onde vem o σ: implícito, histórico, GARCH? Há *look-ahead* (volatilidade realizada futura)?
- [ ] Há filtros de liquidez e de limites de arbitragem (paridade put-call)?
- [ ] Resultados de Monte Carlo vêm com número de cenários e erro-padrão?

**Aprendizado por reforço**
- [ ] O MDP está definido por completo (estado, ação, recompensa, horizonte)?
- [ ] Os sinais usam só o passado? A normalização e os parâmetros foram estimados só no treino? Há embargo?
- [ ] A recompensa usa preços executáveis e custos?
- [ ] Há várias sementes, com dispersão reportada?
- [ ] O teste é fora da amostra, com janelas independentes, intervalo de confiança e teste estatístico?
- [ ] Há comparação com regras simples que usam os mesmos sinais, e com o oráculo?
- [ ] O artigo mostra o que o agente aprendeu (política, importância dos sinais, exemplos)?

**Neste notebook**, a principal limitação é o **prêmio histórico estimado**. A próxima etapa natural é trocar `estimate_historical_put_premiums` por preços reais (COTAHIST da B3 ou cadeia gravada diariamente pelo MT5) e repetir as seções B1 a B9.

## Limitações
- **Prêmio histórico estimado:** a Parte B não usa preços passados reais de opções; o VRP é tratado como constante no tempo.
- **Dividendos e juros:** o dividend yield é o atual, aplicado ao histórico inteiro; os juros históricos são reais (`^IRX`).
- **Opções americanas:** a Parte B usa o BS europeu para estimar prêmios; a seção A7 mostra que a diferença é pequena para puts fora do dinheiro.
- **Um ativo por vez:** os resultados valem para o ativo escolhido; generalizar exige repetir para vários ativos.
- **Custos de transação** não estão na recompensa.

## Referências (artigos)
- Black, F. e Scholes, M. (1973). The Pricing of Options and Corporate Liabilities. *Journal of Political Economy*, 81(3), 637–654.
- Merton, R. C. (1973). Theory of Rational Option Pricing. *Bell Journal of Economics and Management Science*, 4(1), 141–183.
- Cox, J. C., Ross, S. A. e Rubinstein, M. (1979). Option Pricing: A Simplified Approach. *Journal of Financial Economics*, 7(3), 229–263.
- Bollerslev, T. (1986). Generalized Autoregressive Conditional Heteroskedasticity. *Journal of Econometrics*, 31(3), 307–327.
- Bollerslev, T. (1987). A Conditionally Heteroskedastic Time Series Model for Speculative Prices and Rates of Return. *Review of Economics and Statistics*, 69(3), 542–547.
- Jarque, C. M. e Bera, A. K. (1980). Efficient Tests for Normality, Homoscedasticity and Serial Independence of Regression Residuals. *Economics Letters*, 6(3), 255–259.
- Carr, P. e Wu, L. (2009). Variance Risk Premiums. *Review of Financial Studies*, 22(3), 1311–1341.
- Israelov, R. (2019). Pathetic Protection: The Elusive Benefits of Protective Puts. *Journal of Alternative Investments*, 21(3).
- Buehler, H., Gonon, L., Teichmann, J. e Wood, B. (2019). Deep Hedging. *Quantitative Finance*, 19(8), 1271–1291.
- Lin, L.-J. (1992). Self-improving reactive agents based on reinforcement learning, planning and teaching. *Machine Learning*, 8(3–4), 293–321.
- Mnih, V. et al. (2015). Human-level control through deep reinforcement learning. *Nature*, 518, 529–533.
- van Hasselt, H., Guez, A. e Silver, D. (2016). Deep Reinforcement Learning with Double Q-learning. *AAAI*.
- Huber, P. J. (1964). Robust Estimation of a Location Parameter. *Annals of Mathematical Statistics*, 35(1), 73–101.
- Kingma, D. P. e Ba, J. (2015). Adam: A Method for Stochastic Optimization. *ICLR*.
- Henderson, P. et al. (2018). Deep Reinforcement Learning that Matters. *AAAI*.
- Efron, B. (1979). Bootstrap Methods: Another Look at the Jackknife. *Annals of Statistics*, 7(1), 1–26.
- Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32.
- Friedman, J. H. (2001). Greedy Function Approximation: A Gradient Boosting Machine. *Annals of Statistics*, 29(5), 1189–1232.

*Livro de apoio (não é artigo):* Hull, J. *Options, Futures, and Other Derivatives* (Pearson), para a paridade put-call de opções americanas e as gregas.